### imports


In [ ]:
# Standard library imports
import ast
import csv
import logging
import os
import random
import re
import shutil
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from typing import Tuple, Optional

# Third-party imports
import bs4
import google.generativeai as genai
import mysql.connector
import requests
from bs4 import BeautifulSoup
from bingsearch.bingsearch import BingSearch
from dotenv import load_dotenv
from ebooklib import epub
from googlesearch import search
from lxml import etree
from openai import OpenAI
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.support.ui import WebDriverWait
from urllib.parse import quote_plus, urlparse
# Constants
GOODREADS_URL = 'https://www.goodreads.com'
# List of user agents for rotation
LIST_OF_USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.113 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.90 Safari/537.36',
    'Mozilla/5.0 (Windows NT 5.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.90 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.2; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.90 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/44.0.2403.157 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.3; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.113 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/57.0.2987.133 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/57.0.2987.133 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/55.0.2883.87 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/55.0.2883.87 Safari/537.36',
    'Mozilla/4.0 (compatible; MSIE 9.0; Windows NT 6.1)',
    'Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.1; WOW64; Trident/5.0)',
    'Mozilla/5.0 (Windows NT 6.1; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (Windows NT 6.2; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (Windows NT 10.0; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.0; Trident/5.0)',
    'Mozilla/5.0 (Windows NT 6.3; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.1; Trident/5.0)',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 10.0; Windows NT 6.1; WOW64; Trident/6.0)',
    'Mozilla/5.0 (compatible; MSIE 10.0; Windows NT 6.1; Trident/6.0)',
    'Mozilla/4.0 (compatible; MSIE 8.0; Windows NT 5.1; Trident/4.0; .NET CLR 2.0.50727; .NET CLR 3.0.4506.2152; .NET CLR 3.5.30729)'
]



import ast
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Access sensitive data
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY_CHAT")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")

GEMINI_KEY = os.getenv("GEMINI_KEY")

OX_API_USER = os.getenv("OX_API_USER")
OX_API_PASS = os.getenv("OX_API_PASS")
OX_API_URL = os.getenv("OX_API_URL")

# Example: Initialize OpenAI client
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
CALIBRE_PATH = r"C:\Program Files\Calibre2\ebook-meta.exe"  # Adjust if needed
FETCH_METADATA_PATH = r"C:\Program Files\Calibre2\fetch-ebook-metadata.exe"
import html
import json
import subprocess
import uuid
import datetime
from datetime import datetime

import os
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime
import uuid
import html


# functions


In [ ]:


# ==== MySQL Connection (XAMPP) ====

from typing import Dict, List


conn = mysql.connector.connect(
    host="localhost",     # XAMPP MySQL host
    user="root",          # XAMPP MySQL username
    password="",          # XAMPP MySQL password (empty by default)
    database="final_klaus_ebooks_library"
)

cursor = conn.cursor(dictionary=True)

# Configure Gemini AI with the API key from the environment variable
genai.configure(api_key=GEMINI_KEY)

# Create the model configuration
generation_config = {
    "temperature": 0.7,  # Lower temperature for deterministic responses
    "top_p": 0.95,  # Use nucleus sampling
    "top_k": 40,  # Consider top-k tokens
    "max_output_tokens": 512,  # Limit response length
    "response_mime_type": "text/plain",  # Expect text response
}

# Initialize the model
model = genai.GenerativeModel(
    model_name="gemini-2.0-flash-exp",  # Use the appropriate model name
    generation_config=generation_config,
)

genre_chat_session = model.start_chat(history=[])


def get_goodreads_title(book_title, author_name):
    """
    Passes a book title and author to Gemini AI to get the Goodreads official book link.

    Args:
        book_title (str): The title of the book.
        author_name (str): The name of the author.

    Returns:
        str: The Goodreads official book link or an error message.
    """
    # Start a new chat session
    chat_session = model.start_chat(history=[])

    # Construct the input prompt
    prompt = (
        f"Find the official Goodreads title for the book titled '{book_title}' "
        f"written by the author '{author_name}'. Return only the goodreads title."
    )

    # Send the message to the Gemini model
    response = chat_session.send_message(prompt)

    # Extract the text response
    response_text = response.text.strip()
    if response_text.strip().lower() == book_title.strip().lower():
        return None
    return response_text


def fix_ebook_title_with_gemini(unconfirmed_title: str, author_name: str) -> str | None:
    """
    Use Gemini AI to fix or verify an incorrect or approximate eBook title
    by matching it to the official Goodreads title, given the author.

    Args:
        unconfirmed_title (str): The guessed or unclean eBook title.
        author_name (str): The confirmed name of the author.

    Returns:
        str | None: The corrected official title if found, else None.
    """
    prompt = (
        f"this book called: '{unconfirmed_title}' "
        f"by the author '{author_name}' has an incorrect book title. Please return the correct official title "
        "of the book as it appears on Goodreads. Respond with only the exact title, no links or explanations. and make sure the book title is correct "
    )

    try:
        chat = model.start_chat()
        response = chat.send_message(prompt)
        result = response.text.strip()

        # Optionally avoid returning the same name if not corrected
        if result.lower() == unconfirmed_title.strip().lower():
            return None

        return result

    except Exception as e:
        print(f"[Gemini Error] Failed to fix title: {e}")
        return None


def fix_ebook_title_with_openai(unconfirmed_title: str, author_name: str) -> str | None:
    """
    Use OpenAI GPT to fix or verify an incorrect or approximate eBook title
    by matching it to the official Goodreads title, given the author.
    Args:
        unconfirmed_title (str): The guessed or unclean eBook title.
        author_name (str): The confirmed name of the author.
    Returns:
        str | None: The corrected official title if found, else None.
    """
    messages = [
        {
            'role': 'user',
            'content': f"This book called: '{unconfirmed_title}' "
            f"by the author '{author_name}' has an incorrect book title. Please return the correct official Goodreads title "
            "of the book as it appears on Goodreads.com. Respond with only the exact title, no links or explanations. "
            "Make sure the book title is correct."
        }
    ]

    try:
        completion = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        result = completion.choices[0].message.content.strip()

        # Optionally avoid returning the same name if not corrected
        if result.lower() == unconfirmed_title.strip().lower():
            return None
        return result
    except Exception as e:
        print(f"[OpenAI Error] Failed to fix title: {e}")
        return None


def get_book_genres_with_openai(ebook_title: str, author_name: str) -> str | None:
    """
    Use OpenAI GPT to determine the primary Goodreads genre of a book
    based on its title and author.

    Args:
        ebook_title (str): The title of the eBook.
        author_name (str): The name of the author.

    Returns:
        str | None: The most appropriate genre from Goodreads if found, else None.
    """
    messages = [
        {
            'role': 'user',
            'content': f"The book titled '{ebook_title}' by '{author_name}' is listed on Goodreads. "
            "Please return only the 3 **primary** book genre as it appears on Goodreads "
            "(e.g., Fiction, Romance, Historical Fiction, Thriller, Fantasy, Non-Fiction, etc.). "
            "Do not include explanations, quotes, links, or multiple genres—respond with three genre only."
            "return list separated by ,"
        }
    ]

    try:
        completion = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        genre = completion.choices[0].message.content.strip()

        # Optional sanity check: ensure single-word or hyphenated genre
        if not genre or len(genre.split()) > 3:
            return None

        return genre

    except Exception as e:
        # print(f"[OpenAI Error] Failed to get genre: {e}")
        return None
    
def get_another_genre_with_openai(ebook_title: str, genre: str, author_name: str) -> list[str] | None:
    """
    Use OpenAI GPT to determine an additional Goodreads genre of a book
    based on its title, author, and a primary genre.

    Args:
        ebook_title (str): The title of the eBook.
        genre (str): The primary genre of the eBook.
        author_name (str): The name of the author.

    Returns:
        list[str] | None: A list with the primary genre and one additional genre
                          (e.g., ["Fiction", "Historical Fiction"]) if found,
                          else None.
    """
    messages = [
        {
            'role': 'user',
            'content': f"The book titled '{ebook_title}' by '{author_name}' is listed on Goodreads. "
                       f"It falls under the primary genre of '{genre}'. "
                       "Please suggest another genre that fits this book, based on its content and themes. "
                       "Respond with only one genre, no explanations or additional information."
        }
    ]

    try:
        completion = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        additional_genre = completion.choices[0].message.content.strip()

        # Basic sanity check
        if not additional_genre or len(additional_genre.split()) > 3:
            return [genre]  # just return the primary genre

        # Return both genres as a list
        return [genre, additional_genre]

    except Exception as e:
        print(f"[OpenAI Error] Failed to get additional genre: {e}")
        return [genre]


def get_author_name_with_gemini(unconfirmed_title: str) -> str | None:
    """
    Use Gemini AI to identify the official author of a book given only its title.

    Args:
        unconfirmed_title (str): The possibly incorrect or informal book title.

    Returns:
        str | None: The correct author's name if found, otherwise None.
    """
    prompt = (
        f"A book has the title '{unconfirmed_title}', but the author's name is unknown. "
        f"Please provide the full name of the primary author as listed on Goodreads. "
        "Respond with only the author's full name and nothing else."
    )

    try:
        chat = model.start_chat()
        response = chat.send_message(prompt)
        author_name = response.text.strip()

        # Basic validation
        if not author_name or len(author_name.split()) < 2:
            return None

        return author_name

    except Exception as e:
        print(f"[Gemini Error] Failed to get author: {e}")
        return None


def get_book_genre_with_gemini(ebook_title: str, author_name: str) -> str | None:
    """
    Use Gemini AI to determine the primary Goodreads genre of a book
    based on its title and author.

    Args:
        ebook_title (str): The title of the eBook.
        author_name (str): The name of the author.

    Returns:
        str | None: The most appropriate genre from Goodreads if found, else None.
    """
    prompt = (
        f"The book titled '{ebook_title}' by '{author_name}' is listed on Goodreads. "
        "Please return only the 2 **primary** book genre as it appears on Goodreads "
        "(e.g., Fiction, Romance, Historical Fiction, Thriller, Fantasy, Non-Fiction, etc.). "
        "Do not include explanations, quotes, links, or multiple genres—respond with one genre only."
        "return  list separated by ,"
    )

    try:
        response = genre_chat_session.send_message(prompt)
        genre = response.text.strip()

        # Optional sanity check: ensure single-word or hyphenated genre
        if not genre or len(genre.split()) > 3:
            return None

        return genre

    except Exception as e:
        # print(f"[Gemini Error] Failed to get genre: {e}")
        return None


def filter_famous_authors_with_gemini(author_chunks: list[list[str]]) -> list[str]:
    """
    Queries Gemini with chunks of author names and returns only the famous Goodreads authors.

    Args:
        author_chunks (list[list[str]]): A list of chunks, where each chunk contains up to 50 author names.

    Returns:
        list[str]: A flat list of famous Goodreads authors.
    """
    famous_authors = []

    for chunk_idx, chunk in enumerate(author_chunks, start=1):
        # Join authors in this chunk
        authors_str = ", ".join(chunk)

        prompt = (
            f"Here is a list of author names:\n{authors_str}\n\n"
            "Please identify which of these authors are well-known/famous authors "
            "listed on Goodreads. Return only the names of the famous authors "
            "from the list, separated by commas. Do not add explanations."
        )

        try:
            chat = model.start_chat()
            response = chat.send_message(prompt)
            result = response.text.strip()

            if result:
                # Split by comma and clean up whitespace
                famous_list = [name.strip()
                               for name in result.split(",") if name.strip()]
                famous_authors.extend(famous_list)

        except Exception as e:
            print(f"[Gemini Error] Failed on chunk {chunk_idx}: {e}")
            continue

    return famous_authors

# Non-streaming response


def gpt_35_api(messages: list):
    """Create a new response for the provided conversation messages
    Args:
        messages (list): Complete conversation messages
    """
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo", messages=messages)
    print(completion.choices[0].message.content)


def gpt_35_api_stream(messages: list):
    """Create a new response for the provided conversation messages (streaming)
    Args:
        messages (list): Complete conversation messages
    """
    stream = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            print(chunk.choices[0].delta.content, end="")

AUTHORS_CSV = r"updated_authors_final.csv"

def normalize_spaces(text: str) -> str:
    """Collapse multiple spaces into one and strip leading/trailing spaces."""
    return re.sub(r"\s+", " ", text).strip().lower()

def find_author_by_name(author_name, csv_path=AUTHORS_CSV):
    """
    Search for an author by name in updated_authors_final.csv.
    Returns a dict with author details or None if not found.
    """
    if not os.path.isfile(csv_path):
        print(f"❌ File not found: {csv_path}")
        return None

    target = normalize_spaces(author_name)

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row_name = normalize_spaces(row["name"])
            if row_name == target:
                return row  # return the whole row as dict

    return None


def read_metadata_opf(file_path: str) -> Dict[str, Optional[str]]:
    """Read Calibre-style metadata.opf and return a dict of metadata (handles both id= and opf:scheme= identifiers)."""
    ns = {
        "dc": "http://purl.org/dc/elements/1.1/",
        "opf": "http://www.idpf.org/2007/opf"
    }
    tree = ET.parse(file_path)
    root = tree.getroot()

    def gettext(tag: str) -> Optional[str]:
        """Get simple dc:* text field"""
        el = root.find(f".//dc:{tag}", ns)
        return el.text.strip() if el is not None and el.text else None

    def get_identifier(*schemes) -> Optional[str]:
        """Get identifier by any matching scheme (opf:scheme or id attribute)."""
        for scheme in schemes:
            # Try opf:scheme="X"
            el = root.find(f".//dc:identifier[@opf:scheme='{scheme.upper()}']", ns)
            if el is not None and el.text and el.text.strip():
                return el.text.strip()
            # Try id="x"
            el = root.find(f".//dc:identifier[@id='{scheme.lower()}']", ns)
            if el is not None and el.text and el.text.strip():
                return el.text.strip()
        return None

    # Collect all subjects
    subjects: List[str] = [
        el.text.strip()
        for el in root.findall(".//dc:subject", ns)
        if el.text and el.text.strip()
    ]

    # Extract ISBN and GOODREADS identifiers (support both naming styles)
    isbn = get_identifier("ISBN", "isbn")
    goodreads = get_identifier("GOODREADS", "goodreads")

    return {
        "title": gettext("title"),
        "author": gettext("creator"),
        "publisher": gettext("publisher"),
        "language": gettext("language"),
        "description": gettext("description"),
        "date": gettext("date"),
        "subjects": subjects,
        "isbn": isbn,
        "goodreads": goodreads
    }




def try_metadata_opf_fallback(epub_path: str) -> Tuple[
    Optional[str], Optional[str], Optional[str], Optional[str], Optional[List[str]]
]:
    """
    Try to read metadata from metadata.opf (and fallback to meta.opf if needed).
    Returns (book_title, author_name, isbn, goodreads, subjects)
    or (None, None, None, None, None) if nothing found.
    """
    try:
        epub_dir = os.path.dirname(epub_path)
        metadata_opf_path = os.path.join(epub_dir, 'metadata.opf')
        meta_opf_path = os.path.join(epub_dir, 'meta.opf')

        metadata = None

        # --- Step 1: Try metadata.opf ---
        if os.path.exists(metadata_opf_path):
            print(f"📖 Found metadata.opf file: {metadata_opf_path}")
            metadata = read_metadata_opf(metadata_opf_path)

        # --- Step 2: If no metadata.opf, try meta.opf ---
        elif os.path.exists(meta_opf_path):
            print(f"📖 Found meta.opf file: {meta_opf_path}")
            metadata = read_metadata_opf(meta_opf_path)

        else:
            print(f"❌ No metadata.opf or meta.opf found in: {epub_dir}")
            sucess=create_new_meta_opf(epub_dir)
            if sucess:
                print(f"retrying to read metadata from newly created metadata.opf")
                try_metadata_opf_fallback(epub_path)
            

            return None, None, None, None, None

        # Extract key fields
        book_title = metadata.get('title')
        author_name = metadata.get('author')
        isbn = metadata.get('isbn')
        goodreads = metadata.get('goodreads')
        subjects = metadata.get('subjects', [])

        # --- Step 3: If Goodreads missing, check the alternate file ---
        if not goodreads:
            alt_path = (
                meta_opf_path if os.path.exists(meta_opf_path) else metadata_opf_path
            )
            if alt_path != metadata_opf_path and os.path.exists(alt_path):
                print(f"🔄 Goodreads missing, checking alternate OPF: {alt_path}")
                alt_metadata = read_metadata_opf(alt_path)
                goodreads = alt_metadata.get('goodreads', goodreads)
                isbn = isbn or alt_metadata.get('isbn')

        # --- Step 4: Validate ---
        if book_title and author_name:
            genres_str = ", ".join(subjects) if subjects else "N/A"
            print(
                f"✅ Extracted metadata - "
                f"Title: '{book_title}', Author: '{author_name}', "
                f"ISBN: {isbn if isbn else 'N/A'}, Goodreads: {goodreads if goodreads else 'N/A'}, "
                f"Genres: {genres_str}"
            )
            return (
                book_title.strip(),
                author_name.strip(),
                isbn.strip() if isbn else None,
                goodreads.strip() if goodreads else None,
                subjects,
            )

        else:
            print(
                f"⚠️ Incomplete core metadata (Title/Author missing) - "
                f"Title: {book_title}, Author: {author_name}"
            )
            return None, None, None, None, None

    except Exception as e:
        print(f"❌ Error reading metadata.opf: {type(e).__name__} - {e}")
        return None, None, None, None, None

def get_id(bookid):
    pattern = re.compile("([^.-]+)")
    bookdd = bookid.split('-')[0]
    bookdd = bookdd.split('.')[0]
    return bookdd

def get_goodreads_book_id(url: str) -> str:
    """
    Extracts the Goodreads book id (numeric) from a “/book/show/…” URL.
    Returns the id as string, or raises ValueError if not found.
    """
    # Option 1: regex
    # Pattern: /book/show/([0-9]+)
    m = re.search(r'/book/show/(\d+)', url)
    if m:
        return m.group(1)
    
    # Option 2: fallback via path segments
    parsed = urlparse(url)
    path = unquote(parsed.path)  # e.g. "/book/show/7445.The_Glass_Castle"
    parts = path.split('/')
    # find “show” then next part
    for i, part in enumerate(parts):
        if part == "show" and i + 1 < len(parts):
            # next segment might have “{id}.{title}”
            seg = parts[i + 1]
            # split by non-digit
            m2 = re.match(r'^(\d+)', seg)
            if m2:
                return m2.group(1)
    raise ValueError(f"Could not parse Goodreads book id from URL: {url}")

def get_genre_list(soup,book_title, author_name):
    try:
        genres = []
        genres_container = soup.find("div", {"data-testid": "genresList"})
        more_button = genres_container.find(
            "button", string=lambda t: t and "...more" in t)
        # if more_button:
        # print("⚠️ Detected a '...more' button: some genres may be hidden (JS required).")
        # return get_genre_list_with_selenium(book_url)
        if genres_container:
            genre_links = genres_container.find(
                "ul").find("span").find_all("a")
            for link in genre_links:
                genre = link.find("span").text
                genres.append(genre)
            if len(genres) > 1:
                return ', '.join(genres)
            elif len(genres) == 1:
                genre2=get_book_genres_with_openai(book_title, author_name)
                if genre2:
                    #genres.append(genre2)
                    return genre2
                
        return None

    except AttributeError as e:
        print(f"AttributeError in get_genre_list: {e}")
        genre2=get_book_genres_with_openai(book_title, author_name)
        if genre2:
            #genres.append(genre2)
            return genre2
        return None
    except Exception as e:
        print(f"Unexpected error in get_genre_list: {e}")
        genre2=get_book_genres_with_openai(book_title, author_name)
        if genre2:
            #genres.append(genre2)
            return genre2
        return None





def load_combined_map(filepath="combined_map.txt"):
    """Load combined_map dictionary from a text file safely."""
    if not os.path.exists(filepath):
        print(f"⚠️ combined_map file not found: {filepath}")
        return {}
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read().strip()
    try:
        return ast.literal_eval(content)  # Safe evaluation
    except Exception as e:
        print(f"❌ Failed to parse combined_map: {e}")
        return {}

def find_sub_category(genre_list, cat_name=None, map_file="combined_map.txt"):
    """
    Find subcategory id based on genre_list and combined_map file.
    If not found:
      - Use first genre as category (create if needed)
      - Use second genre as subcategory (combine with first if second is single word)
    Returns: (cat_id, sub_id) or (None, None) if nothing found
    """
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="final_klaus_ebooks_library"
    )
    cursor = conn.cursor(dictionary=True)

    # Default values so return never fails
    cat_id, sub_id = None, None  

    # Normalize genre_list
    if isinstance(genre_list, str):
        genre_list = [g.strip().lower() for g in genre_list.split(",")]
    else:
        genre_list = [g.strip().lower() for g in genre_list]

    # Replace special names
    replacements = {"dystopia": "dystopian", "young adult": "young adult"}
    genre_list = [replacements.get(g, g) for g in genre_list]

    # Load combined_map dynamically
    combined_map = load_combined_map(map_file)

    # Fetch existing subcategories
    cursor.execute("SELECT * FROM sub_categories")
    subcategories = cursor.fetchall()

    # 1️⃣ Check combined genres first
    combined_genre = None
    for g1 in genre_list:
        g1=g1.replace('.','').strip()
        for g2 in genre_list:
            g2=g2.replace('.','').strip()
            if g1 != g2 and (g1, g2) in combined_map:
                combined_genre = combined_map[(g1, g2)].lower()
                break
            elif g1 != g2 and (g2, g1) in combined_map:
                combined_genre = combined_map[(g2, g1)].lower()
                break
        if combined_genre:
            break

    if combined_genre:
        for sub in subcategories:
            if combined_genre == sub['sub_category_name'].strip().lower():
                print(f"✅ Found combined subcategory: {sub['sub_category_name']}")
                cat_id, sub_id = sub['cat_id'], sub['id']
                cursor.close()
                conn.close()
                return cat_id, sub_id
            
            
    # 2️⃣ Check individual genres
    for genre in genre_list:
        for sub in subcategories:
            if genre.replace('.','').strip() == sub['sub_category_name'].strip().lower():
                print(f"✅ Found subcategory: {sub['sub_category_name']}")
                cat_id, sub_id = sub['cat_id'], sub['id']
                cursor.close()
                conn.close()
                return cat_id, sub_id

    if combined_genre:
        for sub in subcategories:
            if combined_genre == sub['sub_category_name'].strip().lower():
                print(f"✅ Found combined subcategory: {sub['sub_category_name']}")
                cat_id, sub_id = sub['cat_id'], sub['id']
                cursor.close()
                conn.close()
                return cat_id, sub_id

    cursor.close()
    conn.close()

    # No match found
    return cat_id, sub_id


def get_author_id(soup):
    """
    Retrieves one or more author IDs from a Goodreads book page.

    Args:
        soup (bs4.BeautifulSoup): The BeautifulSoup object representing the HTML page.

    Returns:
        str: A single author ID or multiple author IDs separated by commas.
    """
    author_ids = []
    contributors_section = soup.find('div', class_='ContributorLinksList')

    if contributors_section:
        author_links = contributors_section.find_all(
            'a', class_='ContributorLink')
        for link in author_links:
            author_url = link.get('href', '')
            if '/author/show/' in author_url:
                author_id = author_url.split('/')[-1].split('.')[0]
                #return author_id
                author_ids.append(author_id)

    return ','.join(author_ids) if author_ids else None

def get_author_name(soup):
    author_name_span = soup.find('span', class_='ContributorLink__name', attrs={
                                 'data-testid': 'name'})
    if author_name_span:
        return author_name_span.text.strip()
    return 'Unknown Author'


def get_book_description(soup):
    try:
        desc = soup.find(
            "div", {"class": "DetailsLayoutRightParagraph__widthConstrained"})
        return desc.get_text().strip() if desc else 'No Description available'
    except AttributeError:
        return 'No Description available'


def get_book_cover(soup):
    cover = soup.find('img', {'class': 'ResponsiveImage'})
    return cover['src'] if cover and 'src' in cover.attrs else 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/65/No-Image-Placeholder.svg/1665px-No-Image-Placeholder.svg.png'

# === Fetch Goodreads page with undetectable Chrome ===
def fetch_soup_with_chrome(url, timeout=7):

    options = uc.ChromeOptions()
    options.add_argument('--headless')  # Enable headless mode
    options.add_argument('--headless=new')  # Use new headless mode
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    
    random_user_agent = random.choice(LIST_OF_USER_AGENTS)
    # Add user agent
    options.add_argument(f'--user-agent={random_user_agent}')
    
    driver_path = r"undetected_chromedriver\undetected_chromedriver.exe" 
    driver = uc.Chrome(
        driver_executable_path=driver_path,
        options=options
    )
    

    try:
        driver.get(url)
        # wait until the genres list appears
        time.sleep(random.uniform(2, 4))
        # Wait for page to be ready
        container_elements = WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )
        # container_elements = WebDriverWait(driver, timeout).until(
        #         EC.presence_of_element_located(
        #             (By.CSS_SELECTOR, "div.BookPage__mainContent div.BookPageMetadataSection")
        #         )
        #     )
         
        if container_elements:
            print("✅ container loaded.")
        else:
            print("⚠️ container NOT found.")
    except Exception as e:
        print(f"⚠️ Timeout waiting for genres list: {e}")
        driver.quit()
        return None

    html = driver.page_source
    driver.quit()
    return BeautifulSoup(html, "html.parser")

def scrape_book(book_url, book_url_local, random_header,genres):
    header = {
        "User-Agent": random.choice(LIST_OF_USER_AGENTS),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp",
        "Referer": "https://www.google.com/"
    }

    # First request
    response = requests.get(book_url, headers=header)
    time.sleep(2)

    # Check for redirect
    if response.url != book_url:
        print(f"🔀 Redirected to: {response.url}")
        # Re-request with the redirected URL
        response = requests.get(response.url, headers=header)
        time.sleep(2)
    else:
        print(f"✅ No redirect, using original URL: {book_url}")

    # Parse final response
    #soup = fetch_soup_with_chrome(book_url, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")
        

    book_title_elem = soup.find('h1', class_='Text Text__title1', attrs={
                                'data-testid': 'bookTitle'})
    book_title = ' '.join(book_title_elem.text.split()
                          ) if book_title_elem else 'Unknown Title'
    



    cat_id = None
    sub_id = None
    a_name = get_author_name(soup)
    genre_string = get_genre_list(soup, book_title, a_name)
    
    if genre_string:
        print(genre_string)
        cat_id, sub_id = find_sub_category(genre_string)
    
    if not cat_id or not sub_id:
        if genres and len(genres) > 2:
            print(genres)
            cat_id, sub_id = find_sub_category(genres)
            if not cat_id:  # No category found
                #raise ValueError(f"No matching category found for genres: {genres}")   
                return None, None
        else:
            #genre1 = genres[0]
            genres3=get_book_genres_with_openai(book_title, a_name)
            print(genres3)
            if not genres3:
                time.sleep(3)
                genres3=get_book_genres_with_openai(book_title, a_name)
                print(genres3)
                if not cat_id:  # No category found
                    #raise ValueError(f"No matching category found for genres: {genres3}")   
                    return None, None 
                cat_id, sub_id = find_sub_category(genres3) 
            
    if not cat_id:  # No category found
        #raise ValueError(f"No matching category found for genres: {genres3}")   
        return None, None 

    bookinfo = {
        'id': get_goodreads_book_id(book_url),
        'cat_id': cat_id,
        'sub_cat_id': sub_id,
        'author_ids': get_author_id(soup),
        'book_access': 'Free',
        'title': book_title,
        'description': get_book_description(soup),
        'image': get_book_cover(soup),
        'url_type': 'local',
        'url': book_url_local,
        'download_enable': '1',
        'book_on_rent': '',
        'book_rent_price': '',
        'book_rent_time': '',
        'featured': '1',
        'status': '1'
    }

    return bookinfo, a_name


BOOKS_CSV = r'books.csv'  # Update this path to your CSV file


def get_book_by_title(title, csv_file=BOOKS_CSV):
    """
    Search books.csv by title (case-insensitive) and return book info as dict.
    Returns None if no match is found.
    """
    if not os.path.exists(csv_file):
        print(f"❌ CSV file not found: {csv_file}")
        return None

    with open(csv_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row['title'].strip().lower() == title.strip().lower():
                # Convert numeric fields to int
                book_info = {
                    'id': int(row['id']),
                    'cat_id': int(row['cat_id']),
                    'sub_cat_id': int(row['sub_cat_id']),
                    'author_ids': row['author_ids'],
                    'book_access': row['book_access'],
                    'title': row['title'],
                    'description': row['description'],
                    'image': row['image'],
                    'url_type': row['url_type'],
                    'url': row['url'],
                    'download_enable': row['download_enable'],
                    'book_on_rent': row['book_on_rent'],
                    'book_rent_price': row['book_rent_price'],
                    'book_rent_time': row['book_rent_time'],
                    'featured': row['featured'],
                    'status': row['status']
                }
                return book_info

    print(f"❌ No book found with title: {title}")
    return None


def get_id_number(author_id):
    pattern = re.compile("([^.-]+)")
    aid = pattern.search(author_id).group()
    author_split = aid.split(".")
    return author_split[0]


def search_google_(query, num_results, driver):
    # Perform a Google search with the specified query and number of results
    driver.get(f"https://www.google.com/search?q={query}&num={num_results}")
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.yuRUbf a')))
    #time.sleep(5)  # Allow more time for all results to load

    links = driver.find_elements(By.CSS_SELECTOR, 'div.yuRUbf a')
    urls = [link.get_attribute("href") for link in links if link.get_attribute("href") and 'http' in link.get_attribute("href")]
    urls = [url for url in urls 
            if 'translate.google.com' not in url 
            and 'en.wikipedia.org' not in url]

    unique_domains = set()
    unique_urls = []
    for url in urls:
        domain = urlparse(url).netloc
        if domain not in unique_domains:
            unique_domains.add(domain)
            unique_urls.append(url)

    return urls

def get_google_url(query):
    options = uc.ChromeOptions()
    options.add_argument('--headless')  # Enable headless mode
    options.add_argument('--headless=new')  # Use new headless mode
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    
    random_user_agent = random.choice(LIST_OF_USER_AGENTS)
    # Add user agent
    #options.add_argument(f'--user-agent={random_user_agent}')
    
    driver_path = r"undetected_chromedriver\undetected_chromedriver.exe" 
    driver = uc.Chrome(
        driver_executable_path=driver_path,
        options=options
    )
    urls = search_google_(query, 5, driver)
    print(f'Found {len(urls)} URLs:')    
    driver.quit()# Close the browser
    return urls
            


def get_author_info(soup):
    container = soup.find('div', attrs={'class': 'rightContainer'})
    author_info = {}
    data_div = container.find('br', attrs={'class': 'clear'})
    while data_div:
        if data_div.name:
            data_class = data_div.get('class')[0]
            if data_class == 'aboutAuthorInfo':
                break
            elif data_class == 'dataTitle':
                key = data_div.text.strip()
                author_info[key] = []
            if data_div.text == 'Born':
                data_div = data_div.next_sibling
                author_info[key].append(data_div.strip())
            elif data_div.text == 'Influences':
                data_div = data_div.next_sibling.next_sibling
                data_items = data_div.find_all('span')[-1].find_all('a')
                for data_a in data_items:
                    author_info[key].append(data_a.text.strip())
            elif data_div.text == 'Member Since':
                data_div = data_div.next_sibling.next_sibling
                author_info[key].append(data_div.text.strip())
            else:
                data_items = data_div.find_all('a')
                for data_a in data_items:
                    author_info[key].append(data_a.text.strip())
        data_div = data_div.next_sibling
    return author_info




def get_author_description(soup, id_number):
    cell = soup.find("span", {"id": f"freeTextContainerauthor{id_number}"})
    return cell.text.strip() if cell else None


def get_author_image(soup, author_name):
    cell = soup.find("img", {"alt": author_name, "itemprop": "image"})
    if cell:
        return cell.attrs.get("src")
    return 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/65/No-Image-Placeholder.svg/1665px-No-Image-Placeholder.svg.png'


def oxylabs_search(query, limit=5):
    """Perform search via Oxylabs Realtime API and return list of URLs."""
    try:
        OX_API_USER='king_klaus_qmHfn'
        OX_API_PASS='kiduyuKLAUS1995='
        OX_API_URL='https://realtime.oxylabs.io/v1/queries'
        payload = {
            'source': 'google_search',
            'query': query,
            'domain': 'com',
            'locale': 'en-us',
            'parse': True,
            'start_page': 1,
            'pages': 1,
            'limit': limit
        }

        response = requests.post(OX_API_URL, auth=(
            OX_API_USER, OX_API_PASS), json=payload)
        if response.status_code != 200:
            print(f"Oxylabs API Error: {response.json()}")
            return []

        data = response.json()
        urls = []
        if 'results' in data and len(data['results']) > 0:
            content = data['results'][0].get('content', {})
            results = content.get('results', {})
            organic_results = results.get('organic', [])
            for result in organic_results:
                url = result.get('url')
                if url:
                    urls.append(url)
        return urls

    except Exception as e:
        print(f"Oxylabs search error: {e}")
        return []


def author_youtube_search(author_name):
    query = f"{author_name} channel youtube"
    results = oxylabs_search(query)
    fallback = None
    for url in results:
        if "youtube.com" in url:
            clean = url.split("?")[0].split("#")[0].rstrip("/")
            parts = clean.split("/")
            if len(parts) == 4:
                return clean + "/"
            if not fallback:
                fallback = clean + "/"
    return fallback or ""


def author_instagram_search(author_name):
    query = f"{author_name} instagram official"
    results = oxylabs_search(query)
    fallback = None
    for url in results:
        if "instagram.com" in url:
            clean = url.split("?")[0].split("#")[0].rstrip("/")
            parts = clean.split("/")
            if len(parts) == 4:
                return clean + "/"
            if not fallback:
                fallback = clean + "/"
    return fallback or ""


def author_facebook_search(author_name):
    query = f"{author_name} facebook official"
    results = oxylabs_search(query)
    fallback = None
    for url in results:
        if "facebook.com" in url:
            clean = url.split("?")[0].split("#")[0].rstrip("/")
            parts = clean.split("/")
            if len(parts) == 4:
                return clean + "/"
            if not fallback:
                fallback = clean + "/"
    return fallback or ""


def author_website_search(author_name):
    query = f"{author_name} official website"
    results = oxylabs_search(query)
    if results:
        return results[0]
    return ""


def insert_author_to_db(author):
    """
    Insert author dictionary into MySQL authors table.
    Skips if author with same ID already exists.
    """
    cursor = conn.cursor()
    if not author:
        print("No author data provided.")
        return False

    # Check if author already exists
    cursor.execute("SELECT id FROM authors WHERE id = %s", (author['id'],))
    if cursor.fetchone():
        print(f"Author ID {author['id']} already exists in database.")
        return True

    insert_query = """
        INSERT INTO authors
        (id, name, info, image, facebook_url, instagram_url, youtube_url, website_url, status)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    data = (
        author['id'],
        author['name'],
        author.get('info', None),
        author.get('image', None),
        author.get('facebook_url', None),
        author.get('instagram_url', None),
        author.get('youtube_url', None),
        author.get('website_url', None),
        int(author.get('status', 1))
    )

    try:
        cursor.execute(insert_query, data)
        conn.commit()
        print(f"✅ Author '{author['name']}' inserted successfully!")
        return True
    except mysql.connector.Error as err:
        print(f"❌ Error inserting author: {err}")
        return False


def get_author_by_id(author_id):
    """
    Search for an author in the CSV file by their ID.
    Returns a dictionary of author info if found, else None.
    """
    AUTHOR_CSV_FILE = 'updated_authors_final.csv'  # Path to your CSV file
    author_id = str(author_id)  # Ensure it's a string for comparison
    with open(AUTHOR_CSV_FILE, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if row['id'] == author_id:
                # Return a dictionary with author info
                return {
                    'id': row['id'],
                    'name': row['name'],
                    'info': row['info'],
                    'image': row['image'],
                    'facebook_url': row['facebook_url'],
                    'instagram_url': row['instagram_url'],
                    'youtube_url': row['youtube_url'],
                    'website_url': row['website_url'],
                    'status': row['status']
                }
    return None


def scrape_author(author_id):
    """
    Scrapes the author information from the Goodreads website.

    Args:
        author_id (str): The author ID.

    Returns:
        dict: A dictionary containing the scraped author information.
    """
    user_agent = random.choice(
        LIST_OF_USER_AGENTS)  # Select a random user agent
    # Create a header with the selected user agent
    random_header = {'User-Agent': user_agent}

    url = "https://www.goodreads.com/author/show/" + author_id

    time.sleep(3)  # Pause execution for 3 seconds

    try:
        # Open the URL and retrieve the HTML source with the header
        source = requests.get(url, headers=random_header)
        # Create a BeautifulSoup object for parsing the HTML
        #soup = fetch_soup_with_chrome(url, timeout=7)
        soup = BeautifulSoup(source, "html.parser")

        # Extract the author name from the HTML
        author_name = soup.find("span", {"itemprop": "name"}).text.strip()
        # Call a helper function to get the ID number
        id_number = get_id_number(author_id)

        # Call a helper function to get additional author information
        author_info = get_author_info(soup)
        if author_info:
            if 'Born' in author_info:
                # Extract the author's city name if available
                author_city_name = author_info["Born"][0]
            else:
                author_city_name = 'No City Name Found'

            if 'Website' in author_info:
                # Extract the author's website if available
                author_website = author_info["Website"]
            else:
                author_website = 'none'
        else:
            author_city_name = 'No City Name Found'
            author_website = 'none'

        # info["author_name"] = author_name  # Add the author name to the 'info' dictionary

        author_des = get_author_description(soup, id_number)
        # Check if author_des is empty (None or empty string)
        if not author_des:
            author_des = "No Author Description"

    except AttributeError as e:
        print(
            f"An AttributeError occurred while scraping author information: {e}")
        return None

    # "id","name","description","image","facebook_url","instagram_url","youtube_url","website_url","status"
    if re.search(r"\s{2,}", author_name):
        print(
            f"  ❗ Author name '{author_name}' contains multiple spaces, cleaning it up.")
        new_author_name = re.sub(r"\s{2,}", " ", author_name).strip()

    else:
        new_author_name = author_name.strip()

    author_facebook = author_facebook_search(author_name) 
    author_instagram = author_instagram_search(author_name) 
    author_youtube = author_youtube_search(author_name) 
    author_website = author_website_search(author_name)
    
    
    # --- Validation check ---
    # Check each value individually
    if not author_facebook:
        raise ValueError(f"No Facebook information found for author: {author_name}")
    if not author_instagram:
        raise ValueError(f"No Instagram information found for author: {author_name}")
    if not author_youtube:
        raise ValueError(f"No YouTube information found for author: {author_name}")
    if not author_website:
        raise ValueError(f"No website information found for author: {author_name}")



    return {
        "id": id_number,
        "name": new_author_name,
        "description": author_des,
        "image": get_author_image(soup, author_name),
        "facebook_url": author_facebook,
        "instagram_url": author_instagram,
        "youtube_url": author_youtube,
        "website_url": author_website,
        "status": '1'
    }


def is_author_id_in_csv(aid, csv_file='authors.csv'):
    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.reader(file)
            for row in reader:
                if row[0] == str(aid):  # Check if the first column matches the author ID
                    return True
        return False
    except FileNotFoundError:
        print(f"File {csv_file} not found.")
        return False


def is_author_id_in_local_csv(aid, csv_file):
    """
    Checks if a given author ID exists in the specified CSV file.

    Args:
        aid (int or str): The author ID to search for.
        csv_file (str): The path to the CSV file where the author ID is stored.

    Returns:
        bool: True if the author ID is found in the CSV file, False otherwise.
    """
    try:
        # Open the CSV file in read mode, specifying UTF-8 encoding for compatibility
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            # Create a CSV reader object to read rows from the file
            reader = csv.reader(file)
            for row in reader:
                # Check if the first column matches the provided author ID
                if row[0] == str(aid):  # Convert 'aid' to string for comparison
                    return True  # Return True if a matching author ID is found
        return False  # Return False if the author ID is not found after reading all rows
    except FileNotFoundError:
        # Handle the case where the CSV file does not exist
        print(f"File {csv_file} not found.")
        return False


def append_author_to_csv(file_name, author_data):
    """
    Appends the scraped author data to a CSV file.

    Args:
        file_name (str): The name of the CSV file.
        author_data (dict): A dictionary containing the scraped author information.
    """
    fieldnames = [
        'id', 'name', 'description', 'image',
        'facebook_url', 'instagram_url', 'youtube_url',
        'website_url', 'status'
    ]

    # Check if the file exists and write header if not
    try:
        with open(file_name, mode='r', newline='', encoding='utf-8') as file:
            pass
    except FileNotFoundError:
        with open(file_name, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()

    # Append author data to the CSV file
    with open(file_name, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writerow(author_data)


def clean_url(url):
    return url.split('?')[0] if '?' in url else url


def simple_google_search(name):
    try:
        search_txt = name + ' goodreads book show'

        payload = {
            'source': 'google_search',
            'query': search_txt,
            'domain': 'com',
            'locale': 'en-us',
            'parse': True,
            'start_page': 1,
            'pages': 1,
            'limit': 10,  # Increased to get more results like the original function
        }

        # Get response from Oxylabs API
        response = requests.request(
            'POST',
            'https://realtime.oxylabs.io/v1/queries',
            auth=('king_klaus_qmHfn', 'kiduyuKLAUS1995='),
            json=payload,
        )

        if response.status_code != 200:
            print(f"API Error - {response.json()}")
            return ''

        response_data = response.json()

        # Navigate through the response structure to get organic results
        if 'results' in response_data and len(response_data['results']) > 0:
            content = response_data['results'][0].get('content', {})
            results = content.get('results', {})
            organic_results = results.get('organic', [])

            # Look for Goodreads book show URLs in the organic results
            for result in organic_results:
                url = result.get('url', '')
                if 'https://www.goodreads.com/book/show/' in url:
                    return url

    except Exception as e:
        print(f"goodreads search error: {e}")

    return ''


def get_cover_images(search_term):
    # Add headers to mimic browser request
    headers = {
        "User-Agent": random.choice(LIST_OF_USER_AGENTS)
    }

    # Format search URL
    search_term1 = search_term+' cover'
    search_url = f'https://www.google.com/search?q={search_term1}&tbm=isch'

    # Get page content
    response = requests.get(search_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all images
    image_urls = set()

    # Look for image URLs in script tags
    for script in soup.find_all('script'):
        if script.string:
            urls = re.findall(
                r'https?://[^"\']+(?:jpg|jpeg|png|gif)', script.string)
            for url in urls:
                if not url.startswith('data:'):
                    image_urls.add(url)

    # Also check <img> tags
    for img in soup.find_all('img'):
        src = img.get('src', '')
        if src.startswith('http') and not src.startswith('data:'):
            image_urls.add(src)

    # Also check metadata
    for meta in soup.find_all('meta'):
        content = meta.get('content', '')
        if content.startswith('http') and any(ext in content.lower() for ext in ['.jpg', '.jpeg', '.png', '.gif']):
            image_urls.add(content)

    image_list = list(image_urls)

    # Check for Amazon image URLs
    amazon_patterns = [
        "https://m.media-amazon.com/images/",
        "https://images-na.ssl-images-amazon.com/images"
    ]

    for url in image_list:
        if any(url.startswith(pattern) for pattern in amazon_patterns):
            return url  # return first Amazon match

    # If no Amazon image, return first image found (if any)
    # If no Amazon image, return first image with a valid extension
    for url in image_list:
        print(url)
        if re.search(r'\.(jpg|jpeg|png|gif)$', url, re.IGNORECASE):
            return url
    return None


def extract_search_query(url):
    parsed_url = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed_url.query)
    return params.get('q', [''])[0]


def remove_extra_spaces(text):
    return ' '.join(text.split())


def remove_parentheses_content(title):
    cleaned_title = re.sub(r'\([^)]*\)', '', title)
    return remove_extra_spaces(cleaned_title)


def normalize_text(text):
    import string
    text = remove_parentheses_content(text)
    text = remove_extra_spaces(text.lower())
    return text.translate(str.maketrans('', '', string.punctuation))


def calculate_similarity_score(search_title, book_title):
    search_words = set(normalize_text(search_title).split())
    book_words = set(normalize_text(book_title).split())
    common_words = {'the', 'a', 'an', 'and', 'or', 'but',
                    'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
    search_words -= common_words
    book_words -= common_words
    if not search_words:
        return 0
    return len(search_words & book_words) / len(search_words | book_words)


def is_author_match(search_author, book_authors):
    search_author_norm = normalize_text(search_author)
    for author in book_authors:
        author_norm = normalize_text(author)
        if search_author_norm == author_norm or \
           search_author_norm in author_norm or \
           author_norm in search_author_norm:
            return True
        if len(search_author_norm.split()) >= 2 and len(author_norm.split()) >= 2:
            if search_author_norm.split()[0] in author_norm and \
               search_author_norm.split()[-1] in author_norm:
                return True
    return False


def parse_filename(filename):
    name_without_ext = filename.replace('.epub', '')
    if '-' in name_without_ext:
        parts = name_without_ext.split('-')
        if len(parts) >= 1:
            return ' - '.join(parts[:-1]).replace('_', '').strip(), parts[-1].replace('_', '').strip()
    return None, None


def clean_title_for_comparison(title):
    if not title:
        return title
    # Remove everything after '(' or ':'
    cleaned = re.split(r'[:\(]', title)[0]
    # Remove extra whitespace
    return ' '.join(cleaned.split()).strip()


def clean_search_query(book_title, author_name):
    """
    Cleans and formats the search query for Goodreads search.

    Args:
        book_title (str): The title of the book.
        author_name (str): The name of the author.

    Returns:
        str: A cleaned search query string suitable for Goodreads search.
    """
    # Remove parentheses and extra content from the book title
    cleaned_title = clean_title_for_comparison(book_title)
    # If author name is unknown or not provided, use only the title
    if author_name.lower().strip() in ['unknown', 'n/a', 'na', 'anonymous', '']:
        book_title = cleaned_title

    # Also check for multiple spaces
    if re.search(r'\s{2,}', book_title) or re.search(r'\s{2,}', author_name):
        print(f"  ❗ Query '{query}' contains multiple spaces, cleaning it up.")
        book_title = re.sub(r'\s+', ' ', book_title)
        author_name = re.sub(r'\s+', ' ', author_name)
        query = f"{book_title} by {author_name}"
        return query.strip()
    
    book_title = re.sub(r'[&<>"|{}\\^`\[\]#%]', ' ', book_title).strip()
    author_name = re.sub(r'[&<>"|{}\\^`\[\]#%]', ' ', author_name).strip()
    
    return f"{book_title} by {author_name}"


def check_file_exists(relative_path, base_path=r"C:\xampp8.2\htdocs\php_web_services_final\public"):
    """
    Check if a file exists by combining relative path with base path

    Args:
        relative_path (str): Relative path like "upload\Stephen Deas\The_King_of_the_Crags_-_Stephen_Deas.epub"
        base_path (str): Base directory path

    Returns:
        dict: Contains 'exists' (bool), 'full_path' (str), 'size_bytes', 'size_mb', etc.
    """
    try:
        # Normalize the paths to handle different slash types
        relative_path = relative_path.replace(
            '/', os.sep).replace('\\', os.sep)

        # Combine base path with relative path
        full_path = os.path.join(base_path, relative_path)

        # Normalize the full path
        normalized_path = os.path.normpath(full_path)

        # Check if file exists
        file_exists = os.path.exists(
            normalized_path) and os.path.isfile(normalized_path)

        size_bytes = None
        size_mb = None

        if file_exists:
            size_bytes = os.path.getsize(normalized_path)
            size_mb = round(size_bytes / (1024 * 1024),
                            2)  # Convert bytes to MB

        return {
            'exists': file_exists,
            'full_path': normalized_path,
            'relative_path': relative_path,
            'size_bytes': size_bytes,
            'size_mb': size_mb
        }

    except Exception as e:
        return {
            'exists': False,
            'full_path': None,
            'relative_path': relative_path,
            'error': str(e),
            'size_bytes': None,
            'size_mb': None
        }


def scrape_goodreads_books(url, author_name, book_title=None):
    try:
        print(f"🔍 Scraping Goodreads for: {book_title} by {author_name} from {url}")
        headers = {
            "User-Agent": random.choice(LIST_OF_USER_AGENTS),
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp",
            "Referer": "https://www.google.com/"
        }
        request = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(request) as response:
            source = response.read()

        soup = BeautifulSoup(source, "html.parser")
        book_containers = soup.find_all(
            'tr', itemtype='http://schema.org/Book')
        if not book_containers:
            book_containers = soup.find_all(
                'tr', {'itemtype': 'http://schema.org/Book'})
        if not book_containers:
            return None

        book_matches = []
        search_query = extract_search_query(url)
        search_title = book_title if book_title else search_query

        for container in book_containers:
            try:
                title_element = container.find('a', class_='bookTitle')
                if not title_element:
                    continue
                title_link = title_element.get("href")
                raw_title = title_element.text.strip()
                if not title_link:
                    continue

                cleaned_title = remove_parentheses_content(raw_title)
                authors = [a.text.strip()
                           for a in container.find_all('a', class_='authorName')]
                complete_book_url = clean_url(GOODREADS_URL + title_link)

                # --- Check author match and title similarity ---
                author_match = is_author_match(author_name, authors)
                title_similarity = calculate_similarity_score(
                    search_title, cleaned_title)
                score = 0.7 * (1.0 if author_match else 0.0) + \
                    0.3 * title_similarity

                # --- Extract rating info ---
                rating_span = container.find('span', class_='minirating')
                avg_rating, total_ratings = 0.0, 0
                if rating_span:
                    try:
                        rating_text = rating_span.text.strip()
                        avg_rating_match = re.search(
                            r'([\d.]+) avg rating', rating_text)
                        total_ratings_match = re.search(
                            r'— ([\d,]+) ratings', rating_text)
                        if avg_rating_match:
                            avg_rating = float(avg_rating_match.group(1))
                        if total_ratings_match:
                            total_ratings = int(
                                total_ratings_match.group(1).replace(',', ''))
                    except Exception:
                        pass

                book_matches.append({
                    'title': raw_title,
                    'cleaned_title': cleaned_title,
                    'authors': authors,
                    'url': complete_book_url,
                    'author_match': author_match,
                    'title_similarity': title_similarity,
                    'score': score,
                    'avg_rating': avg_rating,
                    'total_ratings': total_ratings
                })

            except Exception as e:
                print(f"Error processing container: {e}")

        # --- Sort by total_ratings (descending) ---
        book_matches.sort(key=lambda x: x['total_ratings'], reverse=True)

        # --- Pick the best valid match ---
        for match in book_matches:
            if match['total_ratings'] > 0:
                print(
                    f"✅ Best match by total ratings: '{match['title']}' | Avg Rating: {match['avg_rating']} | Total Ratings: {match['total_ratings']}")
                return match['url']

        # If only one book is available, return it regardless of ratings
        if len(book_matches) == 1:
            match = book_matches[0]
            print(
                f"✅ Only one match available: '{match['title']}' | Avg Rating: {match.get('avg_rating', 0)} | Total Ratings: {match.get('total_ratings', 0)}")
            return match['url']

        return None

    except Exception as e:
        print(f"Error scraping Goodreads search: {e}")
        return None


def scrape_goodreads_books_raw(url, author_name, book_title=None):
    try:
        print(f"🔍 Scraping Goodreads raw for: {book_title} by {author_name} from {url}")
        headers = {
            "User-Agent": random.choice(LIST_OF_USER_AGENTS),
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp",
            "Referer": "https://www.google.com/"
        }
        request = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(request) as response:
            source = response.read()

        soup = BeautifulSoup(source, "html.parser")
        book_containers = soup.find_all(
            'tr', itemtype='http://schema.org/Book')

        if not book_containers:
            book_containers = soup.find_all(
                'tr', {'itemtype': 'http://schema.org/Book'})
        if not book_containers:
            return None

        book_matches = []
        search_query = extract_search_query(url)
        search_title = book_title if book_title else search_query

        for container in book_containers:
            try:
                title_element = container.find('a', class_='bookTitle')
                if not title_element:
                    continue
                title_link = title_element.get("href")
                raw_title = title_element.text.strip()
                if not title_link:
                    continue

                cleaned_title = remove_parentheses_content(raw_title)
                authors = [a.text.strip()
                           for a in container.find_all('a', class_='authorName')]
                complete_book_url = clean_url(GOODREADS_URL + title_link)
                author_match = is_author_match(author_name, authors)
                title_similarity = calculate_similarity_score(
                    search_title, raw_title)
                score = 0.7 * (1.0 if author_match else 0.0) + \
                    0.3 * title_similarity

                # --- Extract rating info ---
                rating_span = container.find('span', class_='minirating')
                avg_rating, total_ratings = 0.0, 0
                if rating_span:
                    try:
                        rating_text = rating_span.text.strip()
                        avg_rating_match = re.search(
                            r'([\d.]+) avg rating', rating_text)
                        total_ratings_match = re.search(
                            r'— ([\d,]+) ratings', rating_text)
                        if avg_rating_match:
                            avg_rating = float(avg_rating_match.group(1))
                        if total_ratings_match:
                            total_ratings = int(
                                total_ratings_match.group(1).replace(',', ''))
                    except Exception:
                        pass

                book_matches.append({
                    'title': raw_title,
                    'cleaned_title': cleaned_title,
                    'authors': authors,
                    'url': complete_book_url,
                    'author_match': author_match,
                    'title_similarity': title_similarity,
                    'score': score,
                    'avg_rating': avg_rating,
                    'total_ratings': total_ratings
                })

            except Exception as e:
                print(f"Error processing container: {e}")

        # --- Sort by total_ratings (descending) ---
        book_matches.sort(key=lambda x: x['total_ratings'], reverse=True)

        # --- Pick the best valid match ---
        for match in book_matches:
            if match['total_ratings'] > 0:
                print(
                    f"✅ Best match by total ratings: '{match['title']}' | Avg Rating: {match['avg_rating']} | Total Ratings: {match['total_ratings']}")
                return match['url']

        # If only one book is available, return it regardless of ratings
        if len(book_matches) == 1:
            match = book_matches[0]
            print(
                f"✅ Only one match available: '{match['title']}' | Avg Rating: {match.get('avg_rating', 0)} | Total Ratings: {match.get('total_ratings', 0)}")
            return match['url']

        return None

    except Exception as e:
        print(f"Error scraping Goodreads books: {e}")
        return None


def search_book_id(book_id, csv_file='books.csv'):
    """
    Search for a specific book ID in the 'tbl_books.csv' file.

    Args:
        csv_file (str): Path to the CSV file containing book data.
        book_id (int or str): The book ID to search for in the CSV file.

    Returns:
        bool: True if the book ID is found, False otherwise.

    Raises:
        FileNotFoundError: If the CSV file cannot be found or opened.


    """

    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)

            # Loop through each row in the CSV
            for row in reader:
                # Check if the current row's 'id' matches the given book_id
                if row['id'] == str(book_id):
                    return True  # Return True if the book_id is found
        return False  # Return False if not found
    except FileNotFoundError:
        print(f"Error: The file {csv_file} was not found.")
        return False


def check_book_id(book_id, csv_file):
    """
    Search for a specific book ID in the 'tbl_books.csv' file.

    Args:
        csv_file (str): Path to the CSV file containing book data.
        book_id (int or str): The book ID to search for in the CSV file.

    Returns:
        bool: True if the book ID is found, False otherwise.

    Raises:
        FileNotFoundError: If the CSV file cannot be found or opened.


    """

    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)

            # Loop through each row in the CSV
            for row in reader:
                # Check if the current row's 'id' matches the given book_id
                if row['id'] == str(book_id):
                    return True  # Return True if the book_id is found
        return False  # Return False if not found
    except FileNotFoundError:
        print(f"Error: The file {csv_file} was not found.")
        return False


def check_and_append_aid(aid, csv_file='aid1.csv'):
    """
    Checks if an author ID exists in the CSV file, and if not, appends it on a new line.
    If the file is empty, adds a header before appending the ID.

    Args:
        aid (str): The author ID to search for.
        csv_file (str): The path to the CSV file (default is 'aid1.csv').

    Returns:
        bool: True if the ID was appended, False if it already existed.
    """
    aid_exists = False

    # Check if the file exists and is empty
    file_exists = os.path.exists(csv_file)
    file_empty = os.path.getsize(csv_file) == 0 if file_exists else True

    # Read the CSV and check if the ID exists
    if not file_empty:
        try:
            with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
                reader = csv.DictReader(file)
                for row in reader:
                    if row['aid'] == str(aid):
                        aid_exists = True
                        break
        except FileNotFoundError:
            print(f"File {csv_file} not found, creating a new one.")

    # If the ID doesn't exist, append it to the file
    if not aid_exists:
        with open(csv_file, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            if file_empty:  # Add header if the file is empty or newly created
                writer.writerow(['aid'])
            writer.writerow([aid])  # Append the ID on a new line
        print(f"Appended ID {aid} to {csv_file}.")
        return True
    else:
        print(f"ID {aid} already exists in {csv_file}.")
        return False


def move_and_rename_epub(source_path, destination_directory, new_name):
    """
    Move an EPUB file from the source path to the destination directory and rename it.

    Args:
        source_path (str): The path to the source EPUB file.
        destination_directory (str): The path to the destination directory.
        new_name (str): The new name for the EPUB file (including .epub extension).

    Returns:
        str: The path to the newly moved and renamed EPUB file.
    """
    if not os.path.isfile(source_path):
        print(f"Source file '{source_path}' does not exist.")
        return None

    if not os.path.isdir(destination_directory):
        print(
            f"Destination directory '{destination_directory}' does not exist.")
        return None

    new_file_path = os.path.join(destination_directory, new_name)

    try:
        shutil.move(source_path, new_file_path)
        print(f"File moved and renamed to '{new_file_path}'.")
        return new_file_path
    except Exception as e:
        print(f"Error moving or renaming file: {e}")
        return None

def copy_and_rename_epub(source_path, destination_directory, new_name):
    """
    Copy an EPUB file from the source path to the destination directory and rename it.

    Args:
        source_path (str): The path to the source EPUB file.
        destination_directory (str): The path to the destination directory.
        new_name (str): The new name for the EPUB file (including .epub extension).

    Returns:
        str: The path to the newly copied and renamed EPUB file.
    """
    if not os.path.isfile(source_path):
        print(f"Source file '{source_path}' does not exist.")
        return None

    if not os.path.isdir(destination_directory):
        print(f"Destination directory '{destination_directory}' does not exist.")
        return None

    new_file_path = os.path.join(destination_directory, new_name)

    try:
        shutil.copy2(source_path, new_file_path)  # Copy instead of move
        print(f"File copied and renamed to '{new_file_path}'.")
        return new_file_path
    except Exception as e:
        print(f"Error copying or renaming file: {e}")
        return None

def get_epub_info(fname):
    """
    Extracts metadata from an EPUB file.

    Args:
        fname (str): The path to the EPUB file.

    Returns:
        dict: A dictionary containing the extracted metadata, including 'title' and 'creator',
              or None if an error occurs.
    """
    ns = {
        # Namespace for the container.xml file
        'n': 'urn:oasis:names:tc:opendocument:xmlns:container',
        'pkg': 'http://www.idpf.org/2007/opf',  # Namespace for the package metadata
        'dc': 'http://purl.org/dc/elements/1.1/'  # Namespace for Dublin Core metadata
    }

    try:
        # Open the EPUB file
        zip = zipfile.ZipFile(fname)

        # Read the content of the container.xml file
        txt = zip.read('META-INF/container.xml')
        tree = etree.fromstring(txt)  # Parse the XML content

        # Extract the path of the contents metafile
        cfname = tree.xpath(
            'n:rootfiles/n:rootfile/@full-path', namespaces=ns)[0]

        # Read the contents metafile
        cf = zip.read(cfname)  # Read the contents metafile
        tree = etree.fromstring(cf)  # Parse the XML content

        # Extract the metadata block
        p = tree.xpath('/pkg:package/pkg:metadata', namespaces=ns)[0]

        # Repackage the data
        res = {}  # Initialize a dictionary to store the metadata
        for s in ['title', 'creator']:
            # Extract the text content of each metadata element
            res[s] = p.xpath(f'dc:{s}/text()', namespaces=ns)[0]

        return res  # Return the metadata dictionary

    except (zipfile.BadZipFile, KeyError, etree.XMLSyntaxError, IndexError) as e:
        # Handle specific exceptions (e.g., invalid EPUB format, missing metadata)
        print(f"Error reading EPUB file '{fname}': {e}")
        return None
    except Exception as e:
        # Handle other unexpected exceptions
        print(f"An unexpected error occurred: {e}")
        return None


def is_book_id_in_books_csv(book_id, csv_file='books.csv'):
    """
    Check if a book ID exists in the specified books CSV file.

    Args:
        book_id (int or str): The book ID to search for.
        csv_file (str): Path to the CSV file containing book data.

    Returns:
        bool: True if the book ID is found, False otherwise.
    """
    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                if row['id'] == str(book_id):
                    return True
        return False
    except FileNotFoundError:
        print(f"Error: The file '{csv_file}' was not found.")
        return False


def book_exists_in_csv(book_id, csv_file):
    """
    Same as is_book_id_in_books_csv but for generic use with explicit csv_file.

    Args:
        book_id (int or str): The book ID to search for.
        csv_file (str): Path to the CSV file containing book data.

    Returns:
        bool: True if found, False otherwise.
    """
    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                if row['id'] == str(book_id):
                    return True
        return False
    except FileNotFoundError:
        print(f"Error: The file '{csv_file}' was not found.")
        return False


def is_author_id_in_authors_csv(aid, csv_file='authors.csv'):
    """
    Check if an author ID exists in the specified authors CSV file.

    Args:
        aid (int or str): Author ID to check.
        csv_file (str): Path to the author CSV file.

    Returns:
        bool: True if the author ID is found, False otherwise.
    """
    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.reader(file)
            for row in reader:
                if row[0] == str(aid):
                    return True
        return False
    except FileNotFoundError:
        print(f"File '{csv_file}' not found.")
        return False


def is_book_in_csv(book_file_url, csv_file):
    """
    Checks if a book with the given file URL is present in the specified CSV file.

    Args:
        book_file_url (str): The file URL of the book to search for.
        csv_file (str): The path to the CSV file.

    Returns:
        bool: True if the book is found in the CSV file, otherwise False.
    """
    try:
        with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)

            if 'url' not in reader.fieldnames:
                print("Error: The CSV file does not contain a 'url' column.")
                return False

            for row in reader:
                if row['url'].strip() == book_file_url.strip():
                    return True

        return False

    except FileNotFoundError:
        print(f"Error: The file '{csv_file}' does not exist.")
        return False
    except Exception as e:
        print(f"An error occurred while reading the CSV: {e}")
        return False


def search_epub_by_name(epub_name, 
                        root_dir=r'upload', 
                        root_dir2=r'C:\xampp8.2\htdocs\php_web_services_final\public\upload'):
    """
    Searches for an EPUB file by name within two root directories and their subdirectories.

    Args:
        epub_name (str): The name of the EPUB file to search for (can be full name or partial match).
        root_dir (str): The first root directory to search in.
        root_dir2 (str): The second root directory to search in.

    Returns:
        str: The full path to the EPUB file if found, otherwise None.
    """
    for root in [root_dir, root_dir2]:
        for dirpath, _, filenames in os.walk(root):
            for file in filenames:
                if file.lower().endswith(".epub") and epub_name.lower() in file.lower():
                    return os.path.join(dirpath, file)

    return None  # Not found


# ==== MySQL Check if Book Exists ====
def check_book_url_id(book_id):
    """
    Checks if a book with the given URL already exists in MySQL.
    """
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM books WHERE id = %s", (book_id,))
    count = cursor.fetchone()[0]
    return count > 0


def check_author_id(author_id: int) -> Tuple[bool, Optional[str]]:
    """
    Checks if an author with the given ID already exists in MySQL.

    Args:
        author_id (int): The author ID to check

    Returns:
        Tuple[bool, Optional[str]]: (author_exists, author_name)
            - author_exists: True if author found, False otherwise
            - author_name: Author's name if found, None otherwise
    """
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT name FROM authors WHERE id = %s", (author_id,))
        result = cursor.fetchone()

        if result:
            author_name = result[0]
            return True, author_name
        else:
            return False, None

    except mysql.connector.Error as e:
        print(f"❌ Database error in check_author_id: {e}")
        return False, None
    finally:
        cursor.close()


# ==== MySQL Insert Function ====
def insert_book_to_mysql(book_data):
    try:
        insert_query = """
            INSERT INTO books (
                id, cat_id, sub_cat_id, author_ids, book_access,
                title, description, image, url_type, url,
                download_enable, book_on_rent, book_rent_price, book_rent_time,
                featured, status
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        cursor = conn.cursor()
        cursor.execute(insert_query, (
            book_data.get("id", ""),
            book_data.get("cat_id", ""),
            book_data.get("sub_cat_id", ""),
            book_data.get("author_ids", ""),
            book_data.get("book_access", ""),
            book_data.get("title", ""),
            book_data.get("description", ""),
            book_data.get("image", ""),
            book_data.get("url_type", ""),
            book_data.get("url", ""),
            book_data.get("download_enable", ""),
            book_data.get("book_on_rent", ""),
            book_data.get("book_rent_price", ""),
            book_data.get("book_rent_time", ""),
            book_data.get("featured", ""),
            book_data.get("status", "")
        ))
        conn.commit()
        print(
            f"📚 Book '{book_data.get('title', '')}' inserted into MySQL successfully.")
    except Exception as e:
        print(f"❌ MySQL insert error: {e}")


def remove_prefix(file_path, prefix_to_remove):
    """Remove a specific prefix from a file path"""
    if file_path.startswith(prefix_to_remove):
        return file_path[len(prefix_to_remove):]
    return file_path


def extract_title_author_from_filename(epub_filename: str) -> tuple[str, str]:
    """
    Extract title and author from an EPUB filename.

    Args:
        epub_filename (str): The EPUB filename (e.g., 'Bathed_in_Blood_-_Alex_Archer.epub')

    Returns:
        tuple[str, str]: A tuple containing (title, author)
    """
    # Remove the .epub extension
    filename_without_ext = epub_filename.replace(
        '.epub', '').replace('.EPUB', '')

    # Split by common separators and try to identify title and author
    # Common patterns: "Title_-_Author", "Title - Author", "Author - Title", "Author_Title"

    if '_-_' in filename_without_ext:
        # Pattern: "Title_-_Author"
        parts = filename_without_ext.split('_-_', 1)
        title = parts[0].replace('_', ' ').strip()
        author = parts[1].replace('_', ' ').strip()
    elif ' - ' in filename_without_ext:
        # Pattern: "Title - Author"
        parts = filename_without_ext.split(' - ', 1)
        title = parts[0].strip()
        author = parts[1].strip()
    elif '_' in filename_without_ext:
        # Try to split by underscores and guess which part is title vs author
        parts = filename_without_ext.split('_')
        # Assume last part or last few parts are author
        if len(parts) >= 3:
            # Look for common author patterns (FirstName LastName)
            author_parts = []
            title_parts = []

            # Simple heuristic: if we have parts that look like names at the end
            for i, part in enumerate(parts):
                if i >= len(parts) - 2 and part.istitle():  # Last 2 parts, capitalized
                    author_parts.append(part)
                else:
                    title_parts.append(part)

            if author_parts:
                title = ' '.join(title_parts).strip()
                author = ' '.join(author_parts).strip()
            else:
                # Fallback: assume last part is author
                title = ' '.join(parts[:-1]).strip()
                author = parts[-1].strip()
        else:
            # Only 1-2 parts, treat first as title, last as author
            title = parts[0].replace('_', ' ').strip()
            author = parts[-1].replace('_',
                                       ' ').strip() if len(parts) > 1 else "Unknown"
    else:
        # No clear separator, return whole filename as title
        title = filename_without_ext.replace('_', ' ').strip()
        author = "Unknown"

    # Clean up the results
    title = title.replace('  ', ' ').strip()
    author = author.replace('  ', ' ').strip()

    return title, author

BOOKS_FOLDER = os.path.join(os.getcwd(), 'books')

def log_skipped_book(title, author, epub_path, reason="Unknown"):
    """
    Log skipped books into a CSV file.
    """
    skipped_csv = os.path.join(BOOKS_FOLDER, "skipped_books.csv")
    file_exists = os.path.isfile(skipped_csv)

    with open(skipped_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Title", "Author", "EPUB Path", "Reason"])  # header
        writer.writerow([title, author, epub_path, reason])

    print(f"📝 Logged skipped book: {title} ({reason})")


def log_added_directory(directory_path):
    """
    Log processed directories into a CSV file.
    """
    processed_csv = os.path.join(BOOKS_FOLDER, "processed_dirs.csv")
    file_exists = os.path.isfile(processed_csv)

    with open(processed_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Directory Path"])  # header
        writer.writerow([directory_path])

    print(f"📂 Logged processed directory: {directory_path}")
    
    
    
def scrape_goodreads_cover(url, author_name, book_title=None):
    try:
        print(f"🔍 Scraping Goodreads for: {book_title} by {author_name} from {url}")
        headers = {
            "User-Agent": random.choice(LIST_OF_USER_AGENTS),
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp",
            "Referer": "https://www.google.com/"
        }
        request = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(request) as response:
            source = response.read()

        soup = BeautifulSoup(source, "html.parser")
        book_containers = soup.find_all('tr', itemtype='http://schema.org/Book')
        if not book_containers:
            book_containers = soup.find_all('tr', {'itemtype': 'http://schema.org/Book'})
        if not book_containers:
            return None

        for container in book_containers:
            cover_img_elem = container.find('img', class_='bookCover')
            if cover_img_elem and 'src' in cover_img_elem.attrs:
                book_cover_url = cover_img_elem['src'].strip()

                # Remove extension
                url_no_ext = book_cover_url.rsplit('.', 1)[0]

                # Remove everything after the last dot
                base_url = url_no_ext.rsplit('.', 1)[0]

                # Reattach extension
                final_url = base_url + ".jpg"

                #print(f"🖼️ Original cover URL: {book_cover_url}")
                print(f"🔧 Normalized cover URL: {final_url}")
                return final_url

        return None

    except Exception as e:
        print(f"❌ Error processing cover: {e}")
        return None

def count_files_in_folder(folder_path):
    """
    Count EPUB and PDF files in a single folder (recursive).
    Returns tuple of (folder_name, count).
    """
    count = 0
    folder_name = folder_path.name
    
    try:
        # Use os.walk for better performance than rglob
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(('.epub', '.pdf')):
                    count += 1
    except (PermissionError, OSError):
        pass  # Skip folders we can't access
    
    return (folder_name, count)

def count_books_in_subfolders(root_path):
    """
    Count EPUB and PDF files in immediate subfolders of root directory.
    Returns folders with more than 30 files, sorted by count (descending).
    Uses parallel processing for speed.
    """
    root = Path(root_path)
    
    if not root.exists():
        print(f"Error: Path '{root_path}' does not exist!")
        return
    
    # Get all immediate subfolders
    subfolders = [item for item in root.iterdir() if item.is_dir()]
    
    print(f"Scanning {len(subfolders)} folders...")
    
    folder_counts = {}
    
    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor(max_workers=8) as executor:
        # Submit all tasks
        futures = {executor.submit(count_files_in_folder, folder): folder 
                   for folder in subfolders}
        
        # Collect results as they complete
        completed = 0
        for future in as_completed(futures):
            folder_name, count = future.result()
            folder_counts[folder_name] = count
            completed += 1
            if completed % 10 == 0:
                print(f"Progress: {completed}/{len(subfolders)} folders scanned...")
    
    print(f"Scan complete!\n")
    
    # Prepare data for CSV with full paths
    csv_data = []
    for folder_name, count in folder_counts.items():
        folder_path = root / folder_name
        csv_data.append({
            'folder_name': folder_name,
            'path': str(folder_path),
            'count': count
        })
    
    # Sort by count (descending)
    csv_data.sort(key=lambda x: x['count'], reverse=True)
    
    # Save to CSV
    import csv
    csv_file = 'books.csv'
    
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['folder_name', 'path', 'count'])
        writer.writeheader()
        writer.writerows(csv_data)
    
    print(f"\n✓ Data saved to '{csv_file}'")
    
    # Filter folders with more than 30 files
    filtered_folders = [item for item in csv_data if item['count'] > 30]
    
    # Print summary
    print(f"\nFolders with more than 30 EPUB/PDF files:")
    print(f"{'Folder Name':<50} {'Count':>10}")
    print("=" * 62)
    
    if filtered_folders:
        for item in filtered_folders:
            print(f"{item['folder_name']:<50} {item['count']:>10}")
        print(f"\nTotal folders with >30 files: {len(filtered_folders)}")
    else:
        print("No folders found with more than 30 files.")
    
    print(f"\nTotal subfolders scanned: {len(csv_data)}")
    print(f"Total EPUB/PDF files: {sum(item['count'] for item in csv_data)}")


def get_author_book_counts():
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="",
            database="final_klaus_ebooks_library"
        )

        cursor = conn.cursor(dictionary=True)

        query = """
            SELECT 
                a.id,
                a.name,
                COUNT(b.id) AS book_count
            FROM authors a
            LEFT JOIN books b 
                ON FIND_IN_SET(a.id, b.author_ids)
            GROUP BY a.id, a.name
            ORDER BY book_count DESC;  -- Sort by most books
        """

        cursor.execute(query)
        results = cursor.fetchall()

        cursor.close()
        conn.close()

        return results

    except mysql.connector.Error as err:
        print("Database Error:", err)
        return []





def create_meta_opf(output_path, data):
    """
    Create a fully valid meta.opf file using metadata_dict OR book_data.
    Automatically merges/normalizes fields between both formats.

    Args:
        output_path (str): Path to save meta.opf file
        data (dict): metadata_dict or book_data

    Returns:
        str: Full path to written OPF file
    """

    # ===== NORMALIZATION =====
    # Handle differences between metadata_dict vs book_data
    merged = {
        "title": data.get("title"),
        "author": data.get("author") or data.get("authors"),
        "publisher": data.get("publisher"),
        "language": data.get("language", "en"),
        "description": data.get("description") or data.get("comments"),
        "series": data.get("series"),
        "series_index": None,
        "genres": data.get("genres") or data.get("tags"),
        "rating": data.get("averageRating") or data.get("rating"),
        "goodreads_id": data.get("goodreads_id"),

        "isbn": data.get("isbn"),
        "isbn13": data.get("isbn13"),
        "identifiers": data.get("identifiers"),

        "publication_date": (
            data.get("publication_date") or
            data.get("pubdate")
        )
    }

    # Extract series index if present ("Series #2")
    if merged["series"] and "#" in merged["series"]:
        parts = merged["series"].split("#")
        merged["series"] = parts[0].strip()
        merged["series_index"] = parts[1].strip()

    # ===== XML NAMESPACES =====
    NSMAP = {
        "dc": "http://purl.org/dc/elements/1.1/",
        "opf": "http://www.idpf.org/2007/opf"
    }
    ET.register_namespace('', NSMAP["opf"])
    ET.register_namespace('dc', NSMAP["dc"])
    ET.register_namespace('opf', NSMAP["opf"])

    # ===== ROOT <package> =====
    package = ET.Element("package", {
        "xmlns": NSMAP["opf"],
        "unique-identifier": "uuid_id",
        "version": "2.0"
    })
    metadata = ET.SubElement(package, "metadata", {
        "xmlns:dc": NSMAP["dc"],
        "xmlns:opf": NSMAP["opf"]
    })

    # ===== Identifiers =====
    uuid_id = str(uuid.uuid4())
    ET.SubElement(metadata, f"{{{NSMAP['dc']}}}identifier",
                  {"opf:scheme": "UUID", "id": "uuid_id"}).text = uuid_id

    # ISBN13
    if merged["isbn13"]:
        ET.SubElement(metadata, f"{{{NSMAP['dc']}}}identifier",
                      {"opf:scheme": "ISBN13"}).text = merged["isbn13"]

    # ISBN
    if merged["isbn"]:
        ET.SubElement(metadata, f"{{{NSMAP['dc']}}}identifier",
                      {"opf:scheme": "ISBN"}).text = merged["isbn"]

    # Goodreads ID
    if merged["goodreads_id"]:
        ET.SubElement(metadata, f"{{{NSMAP['dc']}}}identifier",
                      {"opf:scheme": "GOODREADS"}).text = str(merged["goodreads_id"])

    # Additional identifiers: "goodreads:41641777"
    if merged["identifiers"]:
        if isinstance(merged["identifiers"], dict):
            for scheme, val in merged["identifiers"].items():
                ET.SubElement(metadata, f"{{{NSMAP['dc']}}}identifier",
                              {"opf:scheme": scheme.upper()}).text = str(val)

        elif isinstance(merged["identifiers"], str):
            for ident in merged["identifiers"].split(","):
                if ":" in ident:
                    scheme, val = ident.split(":", 1)
                    ET.SubElement(metadata, f"{{{NSMAP['dc']}}}identifier",
                                  {"opf:scheme": scheme.strip().upper()}).text = val.strip()

    # ===== Title =====
    ET.SubElement(metadata, f"{{{NSMAP['dc']}}}title").text = merged["title"] or "Unknown Title"

    # ===== Author(s) =====
    if merged["author"]:
        for a in merged["author"].split(","):
            a = a.strip()
            creator = ET.SubElement(metadata, f"{{{NSMAP['dc']}}}creator", {
                "opf:file-as": a,
                "opf:role": "aut"
            })
            creator.text = a

    # ===== Publisher =====
    if merged["publisher"]:
        ET.SubElement(metadata, f"{{{NSMAP['dc']}}}publisher").text = merged["publisher"]

    # ===== Language =====
    ET.SubElement(metadata, f"{{{NSMAP['dc']}}}language").text = merged["language"]

    # ===== Publication Date =====
    pubdate = merged["publication_date"]
    if pubdate:
        try:
            # Try to normalize
            pubdate_iso = datetime.fromisoformat(pubdate.replace("Z", "")).isoformat() + "Z"
        except Exception:
            pubdate_iso = pubdate
        ET.SubElement(metadata, f"{{{NSMAP['dc']}}}date").text = pubdate_iso

    # ===== Description (HTML allowed) =====
    if merged["description"]:
        safe_desc = html.escape(merged["description"])
        ET.SubElement(metadata, f"{{{NSMAP['dc']}}}description").text = safe_desc

    # ===== Genres / Subjects =====
    if merged["genres"]:
        for g in merged["genres"].split(","):
            ET.SubElement(metadata, f"{{{NSMAP['dc']}}}subject").text = g.strip()

    # ===== Calibre Series =====
    if merged["series"]:
        ET.SubElement(metadata, "meta", {
            "name": "calibre:series",
            "content": merged["series"]
        })

    if merged["series_index"]:
        ET.SubElement(metadata, "meta", {
            "name": "calibre:series_index",
            "content": merged["series_index"]
        })

    # ===== Rating =====
    if merged["rating"]:
        ET.SubElement(metadata, "meta", {
            "name": "calibre:rating",
            "content": str(merged["rating"])
        })

    # ===== Timestamp =====
    ET.SubElement(metadata, "meta", {
        "name": "calibre:timestamp",
        "content": datetime.utcnow().isoformat() + "Z"
    })

    # ===== Title Sort =====
    ET.SubElement(metadata, "meta", {
        "name": "calibre:title_sort",
        "content": merged["title"]
    })

    # ===== Guide (cover reference) =====
    guide = ET.SubElement(package, "guide")
    ET.SubElement(guide, "reference", {
        "type": "cover",
        "title": "Cover",
        "href": "cover.jpg"
    })

    # ===== Write pretty XML =====
    tree = ET.ElementTree(package)
    ET.indent(tree, space=" ", level=0)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)

    print(f"✔ meta.opf saved: {output_path}")
    return output_path

def find_book_details_with_isbn(json_input):
    """
    Searches for the 'details' section within any 'Book:kca://book/...'
    key in a JSON structure and confirms whether it includes 'isbn'.

    Works with either:
      - a file path to a JSON file, or
      - a Python dict/list object (already parsed JSON).
    """
    # Determine if json_input is a path or an object
    if isinstance(json_input, (str, os.PathLike)):
        if not os.path.isfile(json_input):
            print(f"❌ File not found: {json_input}")
            return None
        with open(json_input, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        data = json_input  # assume it’s already a dict or list

    def search_details(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == "details" and isinstance(value, dict):
                    has_isbn = "isbn" in value
                    has_isbn13 = "isbn13" in value
                    return {
                        "found": True,
                        "has_isbn": has_isbn,
                        "has_isbn13": has_isbn13,
                        "details": value
                    }
                result = search_details(value)
                if result:
                    return result
        elif isinstance(obj, list):
            for item in obj:
                result = search_details(item)
                if result:
                    return result
        return None

    result = search_details(data)
    if result:
        #print("✅ 'details' found.")
        #print(f"Contains 'isbn': {result['has_isbn']}, 'isbn13': {result['has_isbn13']}")
        #print(json.dumps(result["details"], indent=4))
        return result
    else:
        print("❌ No 'details' section found.")
        return None

def find_book_stats_with_rating(json_input):
    """
    Searches for the 'stats' section within any 'Book:kca://book/...' key
    in a JSON structure and confirms whether it includes 'averageRating'.
    Works with either:
      - a file path to a JSON file, or
      - a Python dict/list object (already parsed JSON).
    """
    # Determine if json_input is a path or a JSON object
    if isinstance(json_input, (str, os.PathLike)):
        if not os.path.isfile(json_input):
            print(f"❌ File not found: {json_input}")
            return None
        with open(json_input, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        data = json_input  # already a dict or list

    # Recursive search function
    def search_stats(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == "stats" and isinstance(value, dict):
                    has_avg = "averageRating" in value
                    has_ratings = "ratingsCount" in value
                    return {
                        "found": True,
                        "has_averageRating": has_avg,
                        "has_ratingsCount": has_ratings,
                        "stats": value
                    }
                # Recurse deeper
                result = search_stats(value)
                if result:
                    return result
        elif isinstance(obj, list):
            for item in obj:
                result = search_stats(item)
                if result:
                    return result
        return None

    result = search_stats(data)
    if result:
        #print("✅ 'stats' section found.")
        #print(f"Contains 'averageRating': {result['has_averageRating']}, 'ratingsCount': {result['has_ratingsCount']}")
        #print(json.dumps(result["stats"], indent=4))
        return result
    else:
        print("❌ No 'stats' section found.")
        return None

def find_series_entries(json_input):
    """
    Recursively searches the JSON (or JSON file) for any object where '__typename' == 'Series'.
    Returns a list of matches with their key names and full content.
    """
    # --- Load JSON if path provided ---
    if isinstance(json_input, (str, os.PathLike)):
        if not os.path.isfile(json_input):
            print(f"❌ File not found: {json_input}")
            return []
        with open(json_input, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        data = json_input  # already parsed JSON

    matches = []

    # --- Recursive search ---
    def search_series(obj, parent_key=None):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if isinstance(value, dict) and value.get("__typename") == "Series":
                    matches.append({
                        "key": key,
                        "data": value
                    })
                # Recurse deeper
                search_series(value, key)
        elif isinstance(obj, list):
            for item in obj:
                search_series(item, parent_key)

    search_series(data)

    if matches:
        #print(f"✅ Found {len(matches)} Series entries:")
        #for m in matches:
            #print(f"🔹 Key: {m['key']}")
            #print(json.dumps(m["data"], indent=4))
        return matches
    else:
        #print("❌ No Series entries found.")
        return []


    
def find_book_entries_with_description(json_input):
    """
    Recursively searches the JSON (or JSON file) for any object where:
      - '__typename' == 'Book'
      - contains 'legacyId'
      - contains 'description' or 'description({"stripped":true})'

    Returns a list of matches with their key names and full content.
    """
    # --- Load JSON if a file path is given ---
    if isinstance(json_input, (str, os.PathLike)):
        if not os.path.isfile(json_input):
            print(f"❌ File not found: {json_input}")
            return []
        with open(json_input, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        data = json_input  # already parsed JSON

    matches = []

    # --- Recursive search ---
    def search_books(obj, parent_key=None):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if isinstance(value, dict) and value.get("__typename") == "Book":
                    has_legacy = "legacyId" in value
                    has_description = (
                        "description" in value or
                        'description({"stripped":true})' in value
                    )
                    if has_legacy and has_description:
                        matches.append({
                            "key": key,
                            "data": value
                        })
                # Recurse deeper
                search_books(value, key)
        elif isinstance(obj, list):
            for item in obj:
                search_books(item, parent_key)

    # --- Run the recursive search ---
    search_books(data)

    # --- Report results ---
    # if matches:
    #     print(f"✅ Found {len(matches)} Book entries with description:")
    #     for m in matches:
    #         print(f"🔹 Key: {m['key']}")
    #         print(json.dumps(m["data"], indent=4, ensure_ascii=False))
    # else:
    #     print("❌ No Book entries with description found.")

    return matches

def find_book_genres(json_input):
    """
    Searches for the 'bookGenres' section within any 'Book:kca://book/...' key
    and returns all genre names as a comma-separated string.

    Works with either:
      - a file path to a JSON file, or
      - a Python dict/list object (already parsed JSON).
    """
    # --- Load JSON if path provided ---
    if isinstance(json_input, (str, os.PathLike)):
        if not os.path.isfile(json_input):
            print(f"❌ File not found: {json_input}")
            return None
        with open(json_input, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        data = json_input  # already a dict or list

    # --- Recursive search for bookGenres ---
    def search_genres(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == "bookGenres" and isinstance(value, list):
                    genres = []
                    for item in value:
                        if isinstance(item, dict):
                            genre_info = item.get("genre", {})
                            if isinstance(genre_info, dict):
                                name = genre_info.get("name")
                                if name:
                                    genres.append(name.strip())
                    return {
                        "found": True,
                        "genres": genres,
                        "genres_str": ", ".join(genres)
                    }
                # Recurse deeper
                result = search_genres(value)
                if result:
                    return result
        elif isinstance(obj, list):
            for item in obj:
                result = search_genres(item)
                if result:
                    return result
        return None

    result = search_genres(data)
    if result:
        #print("✅ 'bookGenres' section found.")
        #print(f"Genres: {result['genres_str']}")
        return result
    else:
        print("❌ No 'bookGenres' section found.")
        return None

def scrape_book_meta_opf(book_url):
    header = {
        "User-Agent": random.choice(LIST_OF_USER_AGENTS),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp",
        "Referer": "https://www.google.com/"
    }

    # First request
    response = requests.get(book_url, headers=header)
    time.sleep(2)

    # Check for redirect
    if response.url != book_url:
        print(f"🔀 Redirected to: {response.url}")
        # Re-request with the redirected URL
        response = requests.get(response.url, headers=header)
        time.sleep(2)
        
        # Parse final response
    soup = BeautifulSoup(response.text, 'html.parser')

    book_title_elem = soup.find('h1', class_='Text Text__title1', attrs={
                                'data-testid': 'bookTitle'})
    book_title = ' '.join(book_title_elem.text.split()
                          ) if book_title_elem else 'Unknown Title'
    author_name_span = soup.find('span', class_='ContributorLink__name', attrs={
                                 'data-testid': 'name'})
    
    if author_name_span:
        author_name = author_name_span.text.strip()
    else:
        author_name = 'Unknown Author'
    #print(f"Book Title: {book_title}")

    json_data_elem = soup.find('script', id='__NEXT_DATA__', type='application/json')
    json_data = json.loads(json_data_elem.string) if json_data_elem else {}
    json_results=find_book_details_with_isbn(json_data) 
    json_stats=find_book_stats_with_rating(json_data)
    json_genre = find_book_genres(json_data)
    json_series = find_series_entries(json_data)
    goodreads_id=get_goodreads_book_id(book_url)
    json_description = find_book_entries_with_description(json_data)

    #print(json_stats['stats']['averageRating'], json_stats['stats']['ratingsCount'])
    book_title=book_title if book_title else "Unknown Title"
    isbn=json_results['details']['isbn'] if 'isbn' in json_results['details'] else None
    isbn13=json_results['details']['isbn13'] if 'isbn13' in json_results['details'] else None
    publisher= json_results['details']['publisher'] if 'publisher' in json_results['details'] else None
    language=json_results['details']['language']['name'] if 'language' in json_results['details'] else None
    genres=json_genre['genres_str'] if json_genre else None
    series=json_series[0]['data']['title'] if json_series else None
    date = json_results['details']['publicationTime'] if 'publicationTime' in json_results['details'] else None
    if date:
        date = time.strftime('%Y-%m-%d', time.localtime(date / 1000))
    average_rating = json_stats['stats']['averageRating'] if json_stats else None
    ratings_count = json_stats['stats']['ratingsCount'] if json_stats else None
    description= json_description[0]['data']['description'] if json_description else None
    
    return {
        "title": book_title,
        "author": author_name,
        "description": description,
        "isbn": isbn,
        "isbn13": isbn13,
        "publisher": publisher, 
        "language": language,
        "genres": genres,
        "series": series,
        "averageRating": average_rating,
        "ratingsCount": ratings_count,
        "goodreads_id": goodreads_id,
        "publication_date": date
    }
          

def parse_fetched_metadata(metadata_output):
    """Parse the metadata output to extract individual fields."""
    metadata = {}
    
    if not metadata_output:
        return metadata
    
    lines = metadata_output.split('\n')
    current_field = None
    
    for line in lines:
        line_stripped = line.strip()
        
        if ':' in line and not line.startswith(' ') and not line.startswith('<'):
            parts = line.split(':', 1)
            field = parts[0].strip()
            value = parts[1].strip() if len(parts) > 1 else ''
            
            if field == 'Title':
                metadata['title'] = value
                current_field = None
            elif field == 'Author(s)':
                metadata['authors'] = value
                current_field = None
            elif field == 'Publisher':
                metadata['publisher'] = value
                current_field = None
            elif field == 'Tags':
                metadata['tags'] = value
                current_field = None
            elif field == 'Series':
                metadata['series'] = value
                current_field = None
            elif field == 'Languages':
                metadata['language'] = value
                current_field = None
            elif field == 'Rating':
                metadata['rating'] = value
                current_field = None
            elif field == 'Published':
                metadata['pubdate'] = value
                current_field = None
            elif field == 'Identifiers':
                metadata['identifiers'] = value
                current_field = None
            elif field == 'Comments':
                current_field = 'comments'
                metadata['comments'] = value if value else ''
        elif current_field == 'comments':
            # Continue collecting comments lines (including HTML)
            if metadata['comments']:
                metadata['comments'] += '\n' + line
            else:
                metadata['comments'] = line
    
    # Clean up trailing whitespace from comments
    if 'comments' in metadata:
        metadata['comments'] = metadata['comments'].strip()
    
    return metadata


def fetch_metadata_from_goodreads(title, author):
    """Fetch metadata from Goodreads and other sources using Calibre's fetch tool."""
    try:
        cmd = [
            FETCH_METADATA_PATH,
            '--title', title,
            '--authors', author,
            '--verbose',
            '--timeout', '90',
            '--allowed-plugin', 'Goodreads'
            
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='ignore',
            timeout=90
        )
        
        if result.returncode == 0:
            return result.stdout
        else:
            print(f"Failed to fetch metadata: {result.stderr}")
            return None
            
    except Exception as e:
        print(f"Error fetching metadata: {e}")
        return None

def create_new_meta_opf(epub_dir):    
    """
    Creates a new metadata.opf file in the given EPUB directory if it doesn't exist.
    """
    meta_opf_path = os.path.join(epub_dir, 'meta.opf')
    try:
        #get epub files in directory
        epub_files = [f for f in os.listdir(epub_dir) if f.lower().endswith('.epub')]
        if not epub_files:
            print(f"No EPUB files found in directory: {epub_dir}")
            return False
        if not os.path.exists(meta_opf_path):
            epub_path = os.path.join(epub_dir, epub_files[0])
            filename=os.path.basename(epub_path)
            cleaned_title, author_name = extract_title_author_from_filename(filename)
            cleaned_title = re.split(r"\_", cleaned_title)[0].strip() # Remove anything after underscore in title 
            #print(f"Creating new meta.opf in {epub_dir} with Title: '{cleaned_title}' and Author: '{author_name}'") 
            search_query = f"{cleaned_title} {author_name}" 
            book_link = scrape_goodreads_books(
            GOODREADS_URL + '/search?' +
            urllib.parse.urlencode({'q': search_query, 'search_type': 'books'}),
            author_name, cleaned_title)
            if book_link:
                print(f"Found Goodreads link: {book_link}")
                book_data= scrape_book_meta_opf(book_link)
                #print(book_data)
                sucess=create_meta_opf(meta_opf_path, book_data)
                if sucess:
                    return True
                else:
                    return False
            else:
                #print(f"Creating new meta.opf in {epub_dir} with Title: '{cleaned_title}' and Author: '{author_name}'") 
                meta=fetch_metadata_from_goodreads(cleaned_title, author_name)
                #print(meta)
                metadata_dict = parse_fetched_metadata(meta)
                #print(metadata_dict)
                sucess=create_meta_opf(meta_opf_path, metadata_dict)
                if sucess:
                    return True
                else:
                    return False
    except Exception as e:
        print(f"An error occurred while accessing the directory: {e}")
        return False
    



# ==== CONFIGURATION & SETUP ====

class Config:
    """Centralized configuration for the EPUB processing pipeline"""
    
    # Database
    DB_HOST = "localhost"
    DB_USER = "root"
    DB_PASSWORD = ""
    DB_NAME = "final_klaus_ebooks_library"
    
    # File paths
    START_DIR = r'D:\Novels Library\Final_all_books\Elizabeth Lennox'
    BOOKS_FOLDER = os.path.join(os.getcwd(), 'books')
    CSV_FILE = os.path.join(BOOKS_FOLDER, 'allbooks.csv')
    AUTHOR_ID_CSV = 'aid1.csv'
    UPLOAD_FOLDER = r'upload'
    
    # URLs
    GOODREADS_URL = 'https://www.goodreads.com'
    GOODREADS_BOOK_URL = 'https://www.goodreads.com/book/show/'
    GOODREADS_ISBN_SEARCH = 'https://www.goodreads.com/search?q='
    
    # Threading
    MAX_WORKERS = 15
    USE_THREADING = True
    
    # Search methods in priority order
    SEARCH_ATTEMPTS = [
        ("Goodreads Search", lambda bookname, author, title: scrape_goodreads_books(
            Config.GOODREADS_URL + '/search?' + urllib.parse.urlencode({'q': bookname}),
            author, title
        )),
        ("Raw Goodreads Search", lambda bookname, author, title: scrape_goodreads_books_raw(
            Config.GOODREADS_URL + '/search?' + urllib.parse.urlencode(
                {'q': clean_search_query(title, author)}),
            author, None
        )),
        ("Google Search", lambda bookname, author, title: simple_google_search(bookname))
    ]


class Counter:
    """Thread-safe counter management"""
    
    def __init__(self):
        self.processed = 0
        self.skipped = 0
        self.already_added = 0
        self.lock = Lock()
    
    def increment(self, counter_type):
        """Increment counter and return new value"""
        with self.lock:
            if counter_type == 'processed':
                self.processed += 1
                return self.processed
            elif counter_type == 'skipped':
                self.skipped += 1
                return self.skipped
            elif counter_type == 'already_added':
                self.already_added += 1
                return self.already_added
    
    def get_summary(self):
        """Return all counter values"""
        return {
            'processed': self.processed,
            'skipped': self.skipped,
            'already_added': self.already_added
        }


class DatabaseManager:
    """Centralized database operations with thread-safety"""
    
    def __init__(self):
        self.lock = Lock()
        self.conn = self._connect()
    
    def _connect(self):
        """Establish database connection"""
        return mysql.connector.connect(
            host=Config.DB_HOST,
            user=Config.DB_USER,
            password=Config.DB_PASSWORD,
            database=Config.DB_NAME
        )
    
    def execute_safe(self, func, *args, **kwargs):
        """Execute database operation with thread safety"""
        with self.lock:
            return func(*args, **kwargs)
    
    def check_author_id(self, author_id):
        return self.execute_safe(check_author_id, author_id)
    
    def insert_author(self, author_data):
        return self.execute_safe(insert_author_to_db, author_data)
    
    def check_book_url_id(self, book_id):
        return self.execute_safe(check_book_url_id, book_id)
    
    def insert_book(self, book_info):
        return self.execute_safe(insert_book_to_mysql, book_info)
    
    def search_epub_by_name(self, book_url):
        return self.execute_safe(search_epub_by_name, book_url)
    
    def log_skipped_book(self, title, author, path, reason):
        return self.execute_safe(log_skipped_book, title, author, path, reason)
    
    def close(self):
        self.conn.close()


class Logger:
    """Thread-safe logging"""
    
    def __init__(self):
        self.lock = Lock()
    
    def log(self, *args, **kwargs):
        with self.lock:
            print(*args, **kwargs)


def setup_environment():
    """Initialize all required folders and files"""
    os.makedirs(Config.BOOKS_FOLDER, exist_ok=True)
    if not os.path.exists(Config.CSV_FILE):
        with open(Config.CSV_FILE, mode='w', newline='') as f:
            pass


def process_author(book_info, db_manager, logger):
    """Handle author processing and database insertion"""
    author_id = book_info.get("author_ids", "").strip()
    if not author_id:
        raise ValueError("No author_ids found in book_info")
    
    author_name = None
    author_exists, author_name = db_manager.check_author_id(author_id)
    
    if not author_exists:
        author_data = get_author_by_id(int(author_id))
        if author_data:
            db_manager.insert_author(author_data)
        else:
            author_data = scrape_author(author_id)
            logger.log(f"Scraped author data: {author_data}")
            if author_data:
                db_manager.insert_author(author_data)
    else:
        logger.log(f"Author {author_name} already exists in database.")
    
    return author_name


def sanitize_author_name(author_name):
    """Clean and format author name for directory"""
    if re.search(r"\s{2,}", author_name):
        return re.sub(r"\s{2,}", " ", author_name).strip()
    return author_name.strip()


def process_scraped_book(book_info, epub_path, authors_name, book_url_local, db_manager, logger):
    """Process and store a successfully scraped book"""
    try:
        author_name = process_author(book_info, db_manager, logger)
        
        if not author_name:
            author_name = authors_name
        
        logger.log(f'Book scraped ----> {book_info["id"]}')
        
        # Create author directory
        author_dirname = sanitize_author_name(author_name)
        dest = os.path.join(Config.UPLOAD_FOLDER, author_dirname.replace(' ', '_'))
        os.makedirs(dest, exist_ok=True)
        
        # Copy and insert into database
        new_file_path = copy_and_rename_epub(epub_path, dest, book_url_local)
        book_info['url'] = new_file_path
        db_manager.insert_book(book_info)
        
        return True
    
    except Exception as e:
        logger.log(f"❌ Error during book processing: {type(e).__name__} - {e}")
        return False


def scrape_and_process_book(goodreads_url, epub_path, book_url_local, book_meta, db_manager, logger):
    """
    Unified function to scrape and process a book from any source
    
    Returns: (success, processed, already_exists)
    """
    try:
        headers = {'User-Agent': random.choice(LIST_OF_USER_AGENTS)}
        book_info, author_folder = scrape_book(goodreads_url, book_url_local, headers, book_meta.get('genres'))
        
        if not book_info:
            logger.log(f"❌ No book info found from URL: {goodreads_url}")
            return False, False, False
        
        # Check if book already exists
        if not db_manager.check_book_url_id(book_info['id']):
            success = process_scraped_book(book_info, epub_path, author_folder, book_url_local, db_manager, logger)
            if success:
                logger.log(f'✅ Processing {book_meta["name"]} completed!')
                return True, True, False
            else:
                logger.log(f"❌ Failed to process scraped book from '{book_meta['method']}'")
                return False, False, False
        else:
            logger.log(f"🔄 {book_info['title']} already in database, deleting file...")
            try:
                #os.remove(epub_path)
                #Counter.increment('already_added')
                logger.log(f"📂 Deleted file: {epub_path}")
            except Exception as e:
                logger.log(f"⚠️ Could not delete file {epub_path}: {e}")
            return False, False, True
    
    except (KeyError, AttributeError, ConnectionError, TimeoutError, requests.RequestException) as e:
        logger.log(f"❌ Error scraping book from '{book_meta['method']}': {type(e).__name__} - {e}")
        return False, False, False


def search_goodreads_by_isbn(isbn, logger):
    """Search Goodreads by ISBN and return result"""
    try:
        isbn_search_url = Config.GOODREADS_ISBN_SEARCH + urllib.parse.quote(isbn)
        logger.log(f"🔍 Searching Goodreads with ISBN: {isbn_search_url}")
        
        headers = {
            "User-Agent": random.choice(LIST_OF_USER_AGENTS),
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp",
            "Referer": "https://www.google.com/"
        }
        
        response = requests.get(isbn_search_url, headers=headers)
        time.sleep(2)
        
        # Handle redirects
        if response.url != isbn_search_url:
            logger.log(f"🔀 Redirected to: {response.url}")
            response = requests.get(response.url, headers=headers)
            time.sleep(2)
        
        final_url = response.url
        
        # Determine URL type
        if final_url.startswith(Config.GOODREADS_BOOK_URL):
            logger.log("📖 Found direct book page!")
            tag = "goodreads book url"
        elif final_url.startswith(Config.GOODREADS_ISBN_SEARCH):
            logger.log("🔍 Still on search results page")
            tag = "goodreads search url"
        else:
            logger.log(f"🤔 Unexpected URL format: {final_url}")
            tag = "unknown goodreads url"
        
        return {"link": final_url, "tag": tag}
    
    except Exception as e:
        logger.log(f"❌ Error searching by ISBN: {type(e).__name__} - {e}")
        return None


def attempt_book_search(book_meta, epub_path, db_manager, logger, counter):
    """Try all search methods until one succeeds"""
    
    # Priority 1: Direct Goodreads ID
    if book_meta.get('goodreads_id'):
        direct_url = Config.GOODREADS_BOOK_URL + book_meta['goodreads_id']
        logger.log(f"🎯 Found Goodreads ID: {book_meta['goodreads_id']}")
        logger.log(f"🔍 Trying direct URL: {direct_url}")
        
        book_meta['method'] = "Direct Goodreads ID"
        success, processed, already_exists = scrape_and_process_book(
            direct_url, epub_path, book_meta['url_local'], book_meta, db_manager, logger
        )
        if success and processed:
            counter.increment('processed')
            return True
        if already_exists:
            counter.increment('already_added')
            return True  # Already added, no further action needed
    
    # Priority 2: ISBN search
    if book_meta.get('isbn'):
        logger.log(f"📚 Found ISBN: {book_meta['isbn']}")
        isbn_result = search_goodreads_by_isbn(book_meta['isbn'], logger)
        
        if isbn_result and isbn_result.get("link"):
            book_meta['method'] = f"ISBN Search ({isbn_result['tag']})"
            
            if isbn_result["tag"] == "goodreads book url":
                success, processed, _ = scrape_and_process_book(
                    isbn_result["link"], epub_path, book_meta['url_local'], book_meta, db_manager, logger
                )
                if success and processed:
                    counter.increment('processed')
                    return True
            else:
                # Try to parse search results
                try:
                    best_url = scrape_goodreads_books(isbn_result["link"], book_meta['author'], book_meta['title'])
                    if best_url:
                        book_meta['method'] = "ISBN Search -> Book Match"
                        success, processed, _ = scrape_and_process_book(
                            best_url, epub_path, book_meta['url_local'], book_meta, db_manager, logger
                        )
                        if success and processed:
                            counter.increment('processed')
                            return True
                except Exception as e:
                    logger.log(f"⚠️ Error parsing search results: {type(e).__name__} - {e}")
    
    # Priority 3: General search methods
    for attempt_num, (label, search_func) in enumerate(Config.SEARCH_ATTEMPTS, 1):
        logger.log(f"🔍 Attempt {attempt_num}: Trying {label}")
        
        try:
            search_url = search_func(book_meta['name'], book_meta['author'], book_meta['title'])
            if not search_url:
                logger.log(f"❌ {label} failed")
                continue
            
            logger.log(f"✅ {label} found URL: {search_url}")
            book_meta['method'] = label
            
            success, processed, _ = scrape_and_process_book(
                search_url, epub_path, book_meta['url_local'], book_meta, db_manager, logger
            )
            if success and processed:
                counter.increment('processed')
                return True
        
        except (ConnectionError, TimeoutError, requests.RequestException, Exception) as e:
            logger.log(f"⚠️ Error during {label}: {type(e).__name__}")
            continue
    
    return False


def process_single_epub(epub_data, db_manager, logger, counter):
    """Process a single EPUB file"""
    epub_path, file, current_book, total_files = epub_data
    
    logger.log(f"\n🔍 Processing EPUB {current_book} of {total_files}: {file}")
    
    # Extract metadata
    book_title, author_name, isbn, goodreads_id, genres = try_metadata_opf_fallback(epub_path)
    
    if not all([book_title, author_name, isbn, goodreads_id]):
        book_title, author_name = parse_filename(file)
    
    # Prepare book metadata
    book_url_local = file.replace(' ', '_')
    book_name = f"{book_title} {author_name}"
    
    logger.log(f'📖 Ebook {current_book} of {total_files}: {book_name}')
    
    # Check if already processed
    if db_manager.search_epub_by_name(book_url_local):
        logger.log(f"🔄 {book_name} already in database, skipping...")
        counter.increment('already_added')
        return
    
    # Prepare search metadata
    book_meta = {
        'title': book_title,
        'author': author_name,
        'name': book_name,
        'isbn': isbn,
        'goodreads_id': goodreads_id,
        'genres': genres,
        'url_local': book_url_local,
        'current_book': current_book,
        'total_files': total_files
    }
    
    # Attempt to find and process book
    if attempt_book_search(book_meta, epub_path, db_manager, logger, counter):
        return
    
    # All methods failed
    logger.log(f"❌ All search methods failed for '{book_name}'")
    counter.increment('skipped')
    db_manager.log_skipped_book(book_title, author_name, epub_path, "All search methods failed")


def main(start_dir=None, use_threading=None):
    """
    Main processing pipeline
    
    Args:
        start_dir (str): Directory containing EPUB files. Defaults to Config.START_DIR
        use_threading (bool): Whether to use parallel processing. Defaults to Config.USE_THREADING
    """
    if start_dir:
        Config.START_DIR = start_dir
    
    use_threading = use_threading if use_threading is not None else Config.USE_THREADING
    
    setup_environment()
    db_manager = DatabaseManager()
    logger = Logger()
    counter = Counter()
    
    start_time = time.time()
    logger.log(f"{'=' * 50}")
    logger.log(f"Starting EPUB processing from: {Config.START_DIR}")
    logger.log(f"{'=' * 50}")
    
    # Collect all EPUB files
    epub_files = []
    for root, _, files in os.walk(Config.START_DIR):
        for file in files:
            if file.lower().endswith(('.epub', '.pdf')):
                epub_path = os.path.join(root, file)
                epub_files.append((epub_path, file))
    
    total_files = len(epub_files)
    logger.log(f"📚 Found {total_files} files to process")
    
    if use_threading and Config.MAX_WORKERS > 1:
        logger.log(f"🔧 Using {Config.MAX_WORKERS} parallel workers\n")
        with ThreadPoolExecutor(max_workers=Config.MAX_WORKERS) as executor:
            tasks = [(epub_path, file, idx + 1, total_files) 
                     for idx, (epub_path, file) in enumerate(epub_files)]
            futures = {executor.submit(process_single_epub, task, db_manager, logger, counter): task 
                      for task in tasks}
            
            for future in as_completed(futures):
                try:
                    future.result()
                except Exception as e:
                    logger.log(f"❌ Worker error: {type(e).__name__} - {e}")
    else:
        logger.log("Sequential processing mode\n")
        for idx, (epub_path, file) in enumerate(epub_files):
            task = (epub_path, file, idx + 1, total_files)
            process_single_epub(task, db_manager, logger, counter)
    
    # Summary
    log_added_directory(Config.START_DIR)
    
    end_time = time.time()
    duration = end_time - start_time
    hours, remainder = divmod(duration, 3600)
    minutes, seconds = divmod(remainder, 60)
    
    summary = counter.get_summary()
    
    logger.log("\n" + "=" * 60)
    logger.log("📊 PROCESSING SUMMARY")
    logger.log("=" * 60)
    logger.log(f"📚 Total files found: {total_files}")
    logger.log(f"✅ Successfully processed: {summary['processed']}")
    logger.log(f"🔄 Already in database: {summary['already_added']}")
    logger.log(f"⚠️  Skipped: {summary['skipped']}")
    
    if total_files > 0:
        success_rate = (summary['processed'] / total_files * 100)
        logger.log(f"📈 Success rate: {success_rate:.1f}%")
    
    logger.log("=" * 60)
    logger.log(f"⏱️  Total time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")
    logger.log("=" * 60)
    
    db_manager.close()



In [ ]:
if __name__ == "__main__":
    start_dir=r'D:\Novels Library\Authors\James Patterson'
    main(start_dir=start_dir)


### edit sub categories


In [ ]:
import mysql.connector

# Example usage
if __name__ == "__main__":
    authors = get_author_book_counts()
    
    for a in authors:
        print(f"{a['name']} → {a['book_count']} books")


In [ ]:
def count_books_by_category_subcategory(conn):
    """
    Count the number of books for each category and sub-category.

    Args:
        conn: MySQL connection object.

    Returns:
        List of tuples: (category_name, sub_category_name, book_count)
    """
    query = """
    SELECT 
        c.category_name,
        sc.sub_category_name,
        COUNT(b.id) AS book_count
    FROM books b
    JOIN categories c ON b.cat_id = c.id
    LEFT JOIN sub_categories sc ON b.sub_cat_id = sc.id
    WHERE b.status = 1  -- count only active books
    GROUP BY c.category_name, sc.sub_category_name
    ORDER BY c.category_name, sc.sub_category_name;
    """

    cursor = conn.cursor()
    cursor.execute(query)
    results = cursor.fetchall()
    cursor.close()
    return results

conn = mysql.connector.connect(
    host="localhost",     # XAMPP MySQL host
    user="root",          # XAMPP MySQL username
    password="",          # XAMPP MySQL password (empty by default)
    database="final_klaus_ebooks_library"
)
# ==== Example usage ====
book_counts = count_books_by_category_subcategory(conn)
for category, sub_category, count in book_counts:
    print(f"{category} / {sub_category}: {count} books")
